In [ ]:
!git clone https://github.com/dvhanh-code/netiquette-multilabel-classification.git
%cd netiquette-multilabel-classification

Cloning into 'netiquette-multilabel-classification'...
remote: Enumerating objects: 302, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 302 (delta 34), reused 66 (delta 17), pack-reused 204 (from 2)
Receiving objects: 100% (302/302), 219.59 MiB | 15.33 MiB/s, done.
Resolving deltas: 100% (64/64), done.
Updating files: 100% (99/99), done.
/content/netiquette-multilabel-classification


In [ ]:
!pip install transformers datasets accelerate sentencepiece \
              scikit-learn iterative-stratification \
              pandas pyarrow

In [ ]:
!pip install gdown

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DATA_PATH = "/content/drive/MyDrive/masterarbeit/unified_final_v1.parquet"

In [ ]:
import pandas as pd

df = pd.read_parquet(DATA_PATH)

print(df.shape)
print(df["split"].value_counts())


(453242, 12)
split
train    426742
test      13250
val       13250
Name: count, dtype: int64


In [ ]:
!mkdir -p data/final

!ln -s \
"/content/drive/MyDrive/masterarbeit/unified_final_v1.parquet" \
"data/final/unified_final_v1.parquet"

In [ ]:
import os
os.remove("data/final/unified_final_v1.parquet")

# Copy thực
import subprocess
result = subprocess.run([
    "cp",
    "/content/drive/MyDrive/masterarbeit/unified_final_v1.parquet",
    "data/final/unified_final_v1.parquet"
], capture_output=True)
print("Copied to local disk ✅")

# Verify
import pandas as pd
df = pd.read_parquet("data/final/unified_final_v1.parquet")
print(f"Rows: {len(df):,}")

Copied to local disk ✅
Rows: 453,242


## **Data** **Analysis** **bold text**


In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = "data/final/unified_final_v1.parquet"  # đường dẫn dataset của bạn
df = pd.read_parquet(DATA_PATH)

LABELS = ["hate_speech", "toxic", "threat", "insult"]

print(f"Tổng số rows: {len(df):,}\n")

# Nếu có cột 'split' để phân train/val/test
if "split" in df.columns:
    for split in ["train", "val", "test"]:
        sub = df[df["split"] == split]
        print(f"── {split.upper()} ({len(sub):,} rows) ──")
        for label in LABELS:
            col = sub[label]
            annotated = col.notna().sum()      # số rows được annotate (không NaN)
            pos = (col == 1).sum()             # số positive
            neg = (col == 0).sum()             # số negative
            print(f"  {label:<12} annotated={annotated:>7,}  pos={pos:>6,}  neg={neg:>7,}")
        print()
else:
    # Không có split — tính trên toàn bộ
    for label in LABELS:
        col = df[label]
        annotated = col.notna().sum()
        pos = (col == 1).sum()
        neg = (col == 0).sum()
        print(f"  {label:<12} annotated={annotated:>7,}  pos={pos:>6,}  neg={neg:>7,}")

Tổng số rows: 453,242

── TRAIN (426,742 rows) ──
  hate_speech  annotated=278,515  pos= 8,637  neg=269,878
  toxic        annotated=340,767  pos=34,936  neg=305,831
  threat       annotated=236,699  pos=   735  neg=235,964
  insult       annotated=286,314  pos=19,372  neg=266,942

── VAL (13,250 rows) ──
  hate_speech  annotated= 13,250  pos= 1,422  neg= 11,828
  toxic        annotated=  2,722  pos=   819  neg=  1,903
  threat       annotated=  4,295  pos=    21  neg=  4,274
  insult       annotated=  7,017  pos= 1,296  neg=  5,721

── TEST (13,250 rows) ──
  hate_speech  annotated= 13,250  pos= 1,422  neg= 11,828
  toxic        annotated=  2,722  pos=   819  neg=  1,903
  threat       annotated=  4,376  pos=    21  neg=  4,355
  insult       annotated=  7,098  pos= 1,296  neg=  5,802



## Mount Drive + Clone Repo + Install

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 1: Mount Drive + Clone repo + Install
# ═══════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/dvhanh-code/netiquette-multilabel-classification
%cd netiquette-multilabel-classification

!pip install transformers torch pandas numpy scikit-learn -q

import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU not available! Enable GPU: Runtime → Change runtime type → L4")
print(f" GPU:  {torch.cuda.get_device_name(0)}")
print(f" VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into 'netiquette-multilabel-classification'...
remote: Enumerating objects: 302, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 302 (delta 34), reused 66 (delta 17), pack-reused 204 (from 2)
Receiving objects: 100% (302/302), 219.59 MiB | 7.14 MiB/s, done.
Resolving deltas: 100% (64/64), done.
Updating files: 100% (99/99), done.
/content/netiquette-multilabel-classification/netiquette-multilabel-classification
 GPU:  Tesla T4
 VRAM: 15.6 GB


## Setup Data

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 2: Setup data
# ═══════════════════════════════════════════════════════════
import os

os.makedirs("data/final", exist_ok=True)

src = "/content/drive/MyDrive/masterarbeit/unified_final_v1.parquet"
dst = "data/final/unified_final_v1.parquet"

if not os.path.exists(dst):
    os.symlink(src, dst)

if not os.path.exists(dst):
    raise FileNotFoundError(f"Dataset not found: {dst}")

import pandas as pd
df = pd.read_parquet(dst)
print(f" Dataset: {len(df):,} rows")
print(df["split"].value_counts())

 Dataset: 453,242 rows
split
train    426742
test      13250
val       13250
Name: count, dtype: int64


## Load Dataset

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_parquet("data/final/unified_final_v1.parquet")

# Hiển thị shape
print("Shape:", df.shape)

# Hiển thị tên cột
print("\nColumns:")
print(df.columns.tolist())

# Hiển thị 5 dòng đầu
print("\nFirst 5 rows:")
display(df.head(5))

Shape: (453242, 12)

Columns:
['text', 'source', 'language', 'split', 'hate_speech', 'toxic', 'threat', 'insult', 'is_gold', 'data_quality', 'labse_score', 'qualitaet']

First 5 rows:


,text,source,language,split,hate_speech,toxic,threat,insult,is_gold,data_quality,labse_score,qualitaet
0,! Achten Sie auf weitere Updates!,jigsaw,de,train,0.0,0.0,0.0,0.0,False,silver,0.932125,gut
1,"! Beachten Sie, dass die Verwendung von subst ...",jigsaw,de,train,0.0,0.0,0.0,0.0,False,silver,0.919323,gut
2,! Bitte entfernen oder ändern Sie diese AfD-Na...,jigsaw,de,train,0.0,0.0,0.0,0.0,False,silver,0.907281,gut
3,"! Erst einen auf ""Alternative"" machen, von den...",rp_mod,de,train,0.0,NaN,0.0,1.0,True,gold,NaN,None
4,"! Hallo, ich oder ich entfernte den Benutzerna...",jigsaw,de,train,0.0,0.0,0.0,0.0,False,silver,0.973967,gut


In [ ]:
# Setup data + output dirs
import os, subprocess

os.makedirs("data/final", exist_ok=True)

# Output is saved to Drive
DRIVE_OUT = "/content/drive/MyDrive/masterarbeit/results/gbert_large_gold_silver"
os.makedirs(DRIVE_OUT, exist_ok=True)
os.makedirs(f"{DRIVE_OUT}/best_model", exist_ok=True)

print(f"Output dir: {DRIVE_OUT}")

# Copy dataset local
subprocess.run([
    "cp",
    "/content/drive/MyDrive/masterarbeit/unified_final_v1.parquet",
    "data/final/unified_final_v1.parquet"
])

import pandas as pd
df = pd.read_parquet("data/final/unified_final_v1.parquet")
print(f" Dataset: {len(df):,} rows")
print(df["split"].value_counts())

Output dir: /content/drive/MyDrive/masterarbeit/results/gbert_large_gold_silver
 Dataset: 453,242 rows
split
train    426742
test      13250
val       13250
Name: count, dtype: int64


## EXPERIMENT E4: deepset/gbert-large 256

Model:        deepset/gbert-large

Mode:         gold_only

Max length:   256

Batch size:   2

LR:           1e-05

Warmup ratio: 0.03

Drive output: /content/drive/MyDrive/
masterarbeit/results/gbert_large_gold_only_256

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 3: Training — gbert-large gold_only max_length=256
# ═══════════════════════════════════════════════════════════
import os, json, random, shutil
import numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader
from transformers import (
    AutoModel, AutoTokenizer, get_linear_schedule_with_warmup)
from src.training.transformer_dataset import (
    LABELS, NetiquetteTransformerDataset,
    load_dataset, print_dataset_summary)
from src.training.losses import MaskedBCEWithLogitsLoss
from src.training.transformer_metrics import (
    compute_multilabel_metrics, tune_thresholds, print_metrics_table)

# ── Config ─────────────────────────────────────────────────
MODEL_NAME   = "deepset/gbert-large"
MODE         = "gold_only"
MAX_LEN      = 256
BATCH        = 2
EPOCHS       = 3
LR           = 1e-5
WARMUP_RATIO = 0.03   # ← fix: warmup nhỏ hơn
LOCAL_OUT    = "results/gbert_large_gold_only_256"
DRIVE_OUT    = "/content/drive/MyDrive/masterarbeit/results/gbert_large_gold_only_256"
DATA_PATH    = "data/final/unified_final_v1.parquet"

# ── Pre-flight checks ──────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not available!")
device = torch.device("cuda")
print(f"✅ GPU:  {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")
print(f"✅ Data: {DATA_PATH}")

os.makedirs(f"{LOCAL_OUT}/best_model", exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

# ── Save config ────────────────────────────────────────────
json.dump({
    "model_name":    MODEL_NAME,  "mode":          MODE,
    "max_length":    MAX_LEN,     "batch_size":    BATCH,
    "epochs":        EPOCHS,      "data_path":     DATA_PATH,
    "local_out":     LOCAL_OUT,   "drive_out":     DRIVE_OUT,
    "loss":          "MaskedBCEWithLogitsLoss",
    "optimizer":     "AdamW",     "learning_rate": LR,
    "weight_decay":  0.01,        "warmup_ratio":  WARMUP_RATIO,  # ← fix
    "seed":          42,
}, open(f"{LOCAL_OUT}/config.json", "w"), indent=2, ensure_ascii=False)

print(f"\nModel:        {MODEL_NAME}")
print(f"Mode:         {MODE}")
print(f"Max length:   {MAX_LEN}")
print(f"Batch size:   {BATCH}")
print(f"LR:           {LR}")
print(f"Warmup ratio: {WARMUP_RATIO}")
print(f"Drive output: {DRIVE_OUT}")

# ── Sync function ──────────────────────────────────────────
def sync_to_drive():
    for fname in ["config.json", "summary_partial.json",
                  "val_metrics_latest.csv", "thresholds.json",
                  "test_metrics.csv", "summary.json"]:
        src = f"{LOCAL_OUT}/{fname}"
        if os.path.exists(src):
            shutil.copy2(src, f"{DRIVE_OUT}/{fname}")
    for fname in os.listdir(LOCAL_OUT):
        if fname.startswith("val_metrics_epoch_") and fname.endswith(".csv"):
            shutil.copy2(f"{LOCAL_OUT}/{fname}", f"{DRIVE_OUT}/{fname}")
    local_best = f"{LOCAL_OUT}/best_model"
    drive_best = f"{DRIVE_OUT}/best_model"
    if os.path.exists(local_best) and os.listdir(local_best):
        os.makedirs(drive_best, exist_ok=True)
        for fname in os.listdir(local_best):
            shutil.copy2(f"{local_best}/{fname}",
                         f"{drive_best}/{fname}")
    print("  ✅ Synced to Drive")

# ── Model ──────────────────────────────────────────────────
class TransformerClassifier(nn.Module):
    def __init__(self, model_name, num_labels=4, dropout=0.2):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(
            self.encoder.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kw["token_type_ids"] = token_type_ids
        out = self.encoder(**kw)
        pooled = out.pooler_output if (
            hasattr(out, "pooler_output")
            and out.pooler_output is not None
        ) else out.last_hidden_state[:, 0]
        return self.classifier(self.dropout(pooled))

# ── Setup ──────────────────────────────────────────────────
random.seed(42); np.random.seed(42)
torch.manual_seed(42); torch.cuda.manual_seed_all(42)

splits    = load_dataset(DATA_PATH, mode=MODE)
print_dataset_summary(splits)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def make_loader(df, shuffle):
    ds = NetiquetteTransformerDataset(df, tokenizer, MAX_LEN)
    return DataLoader(
        ds, batch_size=BATCH, shuffle=shuffle,
        num_workers=0, pin_memory=True)

train_loader = make_loader(splits["train"], True)
val_loader   = make_loader(splits["val"],   False)
test_loader  = make_loader(splits["test"],  False)

model     = TransformerClassifier(MODEL_NAME).to(device)
loss_fn   = MaskedBCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    int(total_steps * WARMUP_RATIO),   # ← fix: dùng WARMUP_RATIO
    total_steps)

print(f"\nSteps/epoch:  {len(train_loader):,}")
print(f"Total steps:  {total_steps:,}")
print(f"Warmup steps: {int(total_steps * WARMUP_RATIO):,}")
print(f"Est. time:    ~{total_steps * 0.1 / 3600:.1f}h on T4")

sync_to_drive()

# ── Training loop ──────────────────────────────────────────
best_val_s, best_epoch = -1.0, -1
epoch_results = []

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'='*60}\nEPOCH {epoch}/{EPOCHS}\n{'='*60}")

    # Train
    model.train()
    total_loss, steps = 0.0, 0
    for step, batch in enumerate(train_loader, 1):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        logits = model(
            batch["input_ids"], batch["attention_mask"],
            batch.get("token_type_ids"))
        loss = loss_fn(logits, batch["labels"], batch["label_mask"])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += loss.item(); steps += 1
        if step % 500 == 0:
            print(f"  step {step}/{len(train_loader):,} "
                  f"loss={total_loss/steps:.4f}")

    train_loss = total_loss / steps
    print(f"\nEpoch {epoch} train loss: {train_loss:.4f}")

    # Validate
    model.eval()
    all_l, all_lb, all_m = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(
                batch["input_ids"], batch["attention_mask"],
                batch.get("token_type_ids"))
            all_l.append(logits.cpu().numpy())
            all_lb.append(batch["labels"].cpu().numpy())
            all_m.append(batch["label_mask"].cpu().numpy())

    # Threshold=0.5 metrics
    val_metrics = compute_multilabel_metrics(
        np.concatenate(all_l), np.concatenate(all_lb),
        np.concatenate(all_m), split_name="val")
    print_metrics_table(f"VAL Epoch {epoch} threshold=0.5", val_metrics)

    # ← Tuned threshold check (TRONG vòng for loop, indent đúng)
    tuned_check = tune_thresholds(
        np.concatenate(all_l),
        np.concatenate(all_lb),
        np.concatenate(all_m),
        metric="s_score")
    print(f"\n  Tuned thresholds: {tuned_check}")
    val_tuned = compute_multilabel_metrics(
        np.concatenate(all_l),
        np.concatenate(all_lb),
        np.concatenate(all_m),
        thresholds=tuned_check,
        split_name="val_tuned")
    print_metrics_table(f"VAL Epoch {epoch} TUNED", val_tuned)

    macro_s = float(
        val_metrics[val_metrics["label"] == "MACRO"]["s_score"].iloc[0])

    val_metrics.to_csv(
        f"{LOCAL_OUT}/val_metrics_epoch_{epoch}.csv", index=False)
    val_metrics.to_csv(
        f"{LOCAL_OUT}/val_metrics_latest.csv",        index=False)

    if macro_s > best_val_s:
        best_val_s, best_epoch = macro_s, epoch
        torch.save(model.state_dict(),
                   f"{LOCAL_OUT}/best_model/pytorch_model.bin")
        tokenizer.save_pretrained(f"{LOCAL_OUT}/best_model")
        print(f"  → New best: S={macro_s:.4f}")

    epoch_results.append({
        "epoch": epoch, "train_loss": train_loss, "val_macro_s": macro_s})
    json.dump({
        "model_name": MODEL_NAME, "mode":        MODE,
        "epoch":      epoch,      "best_epoch":  best_epoch,
        "best_val_s": best_val_s, "epochs_done": epoch_results,
        "status":     "training",
    }, open(f"{LOCAL_OUT}/summary_partial.json", "w"),
       indent=2, ensure_ascii=False)

    sync_to_drive()

# ── Final evaluation ───────────────────────────────────────
print(f"\nBest epoch: {best_epoch}  |  val S={best_val_s:.4f}")

model.load_state_dict(torch.load(
    f"{LOCAL_OUT}/best_model/pytorch_model.bin",
    map_location=device, weights_only=True))
model.eval()

def collect(loader):
    ll, lb, lm = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(
                batch["input_ids"], batch["attention_mask"],
                batch.get("token_type_ids"))
            ll.append(logits.cpu().numpy())
            lb.append(batch["labels"].cpu().numpy())
            lm.append(batch["label_mask"].cpu().numpy())
    return (np.concatenate(ll),
            np.concatenate(lb),
            np.concatenate(lm))

val_logits, val_labels, val_masks = collect(val_loader)
best_thresholds = tune_thresholds(
    val_logits, val_labels, val_masks, metric="s_score")
print("Thresholds:", best_thresholds)

test_logits, test_labels, test_masks = collect(test_loader)
test_metrics = compute_multilabel_metrics(
    test_logits, test_labels, test_masks,
    thresholds=best_thresholds, split_name="test")
print_metrics_table("FINAL TEST METRICS", test_metrics)

test_metrics.to_csv(f"{LOCAL_OUT}/test_metrics.csv", index=False)
json.dump(best_thresholds,
          open(f"{LOCAL_OUT}/thresholds.json", "w"), indent=2)
json.dump({
    "model_name":  MODEL_NAME, "mode":       MODE,
    "max_length":  MAX_LEN,   "batch_size": BATCH,
    "best_epoch":  best_epoch,"best_val_s": best_val_s,
    "epochs_done": epoch_results, "status": "done",
    "test_macro":  test_metrics[
        test_metrics["label"] == "MACRO"].iloc[0].to_dict(),
}, open(f"{LOCAL_OUT}/summary.json", "w"),
   indent=2, ensure_ascii=False)

sync_to_drive()
print(f"\n✅ Done! Results at:\n  {DRIVE_OUT}")

In [ ]:
!git add results/gbert_large_gold_only_256/config.json \
        results/gbert_large_gold_only_256/val_metrics_epoch_*.csv \
        results/gbert_large_gold_only_256/val_metrics_latest.csv \
        results/gbert_large_gold_only_256/test_metrics.csv \
        results/gbert_large_gold_only_256/thresholds.json \
        results/gbert_large_gold_only_256/summary.json

!git commit -m "Add GBERT large gold-only results"

!git push

## bert-base-german-cased 256

MODEL_NAME   = "bert-base-german-cased"
MODE         = "gold_silver"

MAX_LEN      = 256
BATCH        = 4

EPOCHS       = 3

LR           = 2e-5
WARMUP_RATIO = 0.03

LOCAL_OUT = "results/bert_base_gold_silver_256"
DRIVE_OUT = (
    "/content/drive/MyDrive/masterarbeit/"
    "results/bert_base_gold_silver_256"
)

DATA_PATH = "data/final/unified_final_v1.parquet"

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 3: Training — bert-base-german-cased
# gold_silver + max_length=256 + tuned threshold
# FINAL STABLE VERSION
# ═══════════════════════════════════════════════════════════

import os
import json
import random
import shutil

import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import DataLoader

from transformers import (
    AutoModel,
    AutoTokenizer,
    get_linear_schedule_with_warmup
)

from src.training.transformer_dataset import (
    LABELS,
    NetiquetteTransformerDataset,
    load_dataset,
    print_dataset_summary
)

from src.training.losses import MaskedBCEWithLogitsLoss

from src.training.transformer_metrics import (
    compute_multilabel_metrics,
    tune_thresholds,
    print_metrics_table
)

# ───────────────────────────────────────────────────────────
# CONFIG
# ───────────────────────────────────────────────────────────

MODEL_NAME   = "bert-base-german-cased"
MODE         = "gold_silver"

MAX_LEN      = 256
BATCH        = 4

EPOCHS       = 3

LR           = 2e-5
WARMUP_RATIO = 0.03

LOCAL_OUT = "results/bert_base_gold_silver_256"
DRIVE_OUT = (
    "/content/drive/MyDrive/masterarbeit/"
    "results/bert_base_gold_silver_256"
)

DATA_PATH = "data/final/unified_final_v1.parquet"

# ───────────────────────────────────────────────────────────
# PRE-FLIGHT CHECKS
# ───────────────────────────────────────────────────────────

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not available!")

device = torch.device("cuda")

print(f"✅ GPU:  {torch.cuda.get_device_name(0)}")
print(
    f"✅ VRAM: "
    f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

print(f"✅ Data: {DATA_PATH}")

os.makedirs(f"{LOCAL_OUT}/best_model", exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

# ───────────────────────────────────────────────────────────
# SAVE CONFIG
# ───────────────────────────────────────────────────────────

config = {
    "model_name": MODEL_NAME,
    "mode": MODE,
    "max_length": MAX_LEN,
    "batch_size": BATCH,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": 0.01,
    "loss": "MaskedBCEWithLogitsLoss",
    "optimizer": "AdamW",
    "data_path": DATA_PATH,
    "seed": 42,
}

with open(f"{LOCAL_OUT}/config.json", "w") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("\nCONFIG")
print("=" * 60)
print(f"Model:         {MODEL_NAME}")
print(f"Mode:          {MODE}")
print(f"Max length:    {MAX_LEN}")
print(f"Batch size:    {BATCH}")
print(f"Epochs:        {EPOCHS}")
print(f"Learning rate: {LR}")
print(f"Warmup ratio:  {WARMUP_RATIO}")
print(f"Local output:  {LOCAL_OUT}")
print(f"Drive output:  {DRIVE_OUT}")

# ───────────────────────────────────────────────────────────
# SYNC FUNCTION
# ───────────────────────────────────────────────────────────

def sync_to_drive():

    fixed_files = [
        "config.json",
        "summary_partial.json",
        "summary.json",
        "thresholds.json",
        "test_metrics.csv",
        "val_metrics_latest.csv",
        "val_tuned_latest.csv",
    ]

    for fname in fixed_files:

        src = f"{LOCAL_OUT}/{fname}"

        if os.path.exists(src):
            shutil.copy2(src, f"{DRIVE_OUT}/{fname}")

    # sync all epoch metric files

    for fname in os.listdir(LOCAL_OUT):

        if (
            fname.startswith("val_metrics_epoch_")
            or fname.startswith("val_tuned_epoch_")
        ) and fname.endswith(".csv"):

            shutil.copy2(
                f"{LOCAL_OUT}/{fname}",
                f"{DRIVE_OUT}/{fname}"
            )

    # sync best model

    local_best = f"{LOCAL_OUT}/best_model"
    drive_best = f"{DRIVE_OUT}/best_model"

    if os.path.exists(local_best):

        os.makedirs(drive_best, exist_ok=True)

        for fname in os.listdir(local_best):

            shutil.copy2(
                f"{local_best}/{fname}",
                f"{drive_best}/{fname}"
            )

    print("  ✅ Synced to Drive")

# ───────────────────────────────────────────────────────────
# MODEL
# ───────────────────────────────────────────────────────────

class TransformerClassifier(nn.Module):

    def __init__(
        self,
        model_name,
        num_labels=4,
        dropout=0.2
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            self.encoder.config.hidden_size,
            num_labels
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids=None
    ):

        kwargs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        }

        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids

        outputs = self.encoder(**kwargs)

        if (
            hasattr(outputs, "pooler_output")
            and outputs.pooler_output is not None
        ):
            pooled = outputs.pooler_output
        else:
            pooled = outputs.last_hidden_state[:, 0]

        pooled = self.dropout(pooled)

        logits = self.classifier(pooled)

        return logits

# ───────────────────────────────────────────────────────────
# REPRODUCIBILITY
# ───────────────────────────────────────────────────────────

random.seed(42)
np.random.seed(42)

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# ───────────────────────────────────────────────────────────
# LOAD DATA
# ───────────────────────────────────────────────────────────

splits = load_dataset(DATA_PATH, mode=MODE)

print_dataset_summary(splits)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

# ───────────────────────────────────────────────────────────
# DATALOADERS
# ───────────────────────────────────────────────────────────

def make_loader(df, shuffle):

    dataset = NetiquetteTransformerDataset(
        df=df,
        tokenizer=tokenizer,
        max_length=MAX_LEN
    )

    return DataLoader(
        dataset,
        batch_size=BATCH,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=True,
    )

train_loader = make_loader(splits["train"], shuffle=True)
val_loader   = make_loader(splits["val"],   shuffle=False)
test_loader  = make_loader(splits["test"],  shuffle=False)

# ───────────────────────────────────────────────────────────
# MODEL SETUP
# ───────────────────────────────────────────────────────────

model = TransformerClassifier(MODEL_NAME).to(device)

loss_fn = MaskedBCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.01
)

total_steps = len(train_loader) * EPOCHS

warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print("\nTRAINING INFO")
print("=" * 60)
print(f"Steps per epoch: {len(train_loader):,}")
print(f"Total steps:     {total_steps:,}")
print(f"Warmup steps:    {warmup_steps:,}")

sync_to_drive()

# ───────────────────────────────────────────────────────────
# TRAINING LOOP
# ───────────────────────────────────────────────────────────

best_val_s = -1.0
best_epoch = -1

epoch_results = []

for epoch in range(1, EPOCHS + 1):

    print("\n" + "=" * 60)
    print(f"EPOCH {epoch}/{EPOCHS}")
    print("=" * 60)

    # ───────────────────────────────────────────────────────
    # TRAIN
    # ───────────────────────────────────────────────────────

    model.train()

    total_loss = 0.0
    steps = 0

    for step, batch in enumerate(train_loader, start=1):

        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        optimizer.zero_grad()

        logits = model(
            batch["input_ids"],
            batch["attention_mask"],
            batch.get("token_type_ids")
        )

        loss = loss_fn(
            logits,
            batch["labels"],
            batch["label_mask"]
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        steps += 1

        if step % 500 == 0:

            avg_loss = total_loss / steps

            print(
                f"  step {step:,}/{len(train_loader):,} "
                f"loss={avg_loss:.4f}"
            )

    train_loss = total_loss / steps

    print(f"\nEpoch {epoch} train loss: {train_loss:.4f}")

    # ───────────────────────────────────────────────────────
    # VALIDATION
    # ───────────────────────────────────────────────────────

    model.eval()

    all_logits = []
    all_labels = []
    all_masks  = []

    with torch.no_grad():

        for batch in val_loader:

            batch = {
                k: v.to(device)
                for k, v in batch.items()
            }

            logits = model(
                batch["input_ids"],
                batch["attention_mask"],
                batch.get("token_type_ids")
            )

            all_logits.append(logits.cpu().numpy())
            all_labels.append(batch["labels"].cpu().numpy())
            all_masks.append(batch["label_mask"].cpu().numpy())

    val_logits = np.concatenate(all_logits)
    val_labels = np.concatenate(all_labels)
    val_masks  = np.concatenate(all_masks)

    # ───────────────────────────────────────────────────────
    # THRESHOLD = 0.5
    # ───────────────────────────────────────────────────────

    val_metrics = compute_multilabel_metrics(
        val_logits,
        val_labels,
        val_masks,
        split_name="val"
    )

    print_metrics_table(
        f"VAL Epoch {epoch} threshold=0.5",
        val_metrics
    )

    # ───────────────────────────────────────────────────────
    # TUNED THRESHOLDS
    # ───────────────────────────────────────────────────────

    tuned_thresholds = tune_thresholds(
        val_logits,
        val_labels,
        val_masks,
        metric="s_score"
    )

    print(f"\nTuned thresholds: {tuned_thresholds}")

    val_tuned = compute_multilabel_metrics(
        val_logits,
        val_labels,
        val_masks,
        thresholds=tuned_thresholds,
        split_name="val_tuned"
    )

    print_metrics_table(
        f"VAL Epoch {epoch} TUNED",
        val_tuned
    )

    # IMPORTANT:
    # choose best model using tuned thresholds

    macro_s = float(
        val_tuned[
            val_tuned["label"] == "MACRO"
        ]["s_score"].iloc[0]
    )

    # ───────────────────────────────────────────────────────
    # SAVE METRICS
    # ───────────────────────────────────────────────────────

    val_metrics.to_csv(
        f"{LOCAL_OUT}/val_metrics_epoch_{epoch}.csv",
        index=False
    )

    val_tuned.to_csv(
        f"{LOCAL_OUT}/val_tuned_epoch_{epoch}.csv",
        index=False
    )

    val_metrics.to_csv(
        f"{LOCAL_OUT}/val_metrics_latest.csv",
        index=False
    )

    val_tuned.to_csv(
        f"{LOCAL_OUT}/val_tuned_latest.csv",
        index=False
    )

    # ───────────────────────────────────────────────────────
    # SAVE BEST MODEL
    # ───────────────────────────────────────────────────────

    if macro_s > best_val_s:

        best_val_s = macro_s
        best_epoch = epoch

        torch.save(
            model.state_dict(),
            f"{LOCAL_OUT}/best_model/pytorch_model.bin"
        )

        tokenizer.save_pretrained(
            f"{LOCAL_OUT}/best_model"
        )

        print(f"\n  → New best model: S={macro_s:.4f}")

    # ───────────────────────────────────────────────────────
    # SAVE PARTIAL SUMMARY
    # ───────────────────────────────────────────────────────

    epoch_results.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_macro_s": macro_s,
    })

    partial_summary = {
        "model_name": MODEL_NAME,
        "mode": MODE,
        "epoch": epoch,
        "best_epoch": best_epoch,
        "best_val_s": best_val_s,
        "epochs_done": epoch_results,
        "status": "training",
    }

    with open(f"{LOCAL_OUT}/summary_partial.json", "w") as f:

        json.dump(
            partial_summary,
            f,
            indent=2,
            ensure_ascii=False
        )

    sync_to_drive()

# ───────────────────────────────────────────────────────────
# FINAL EVALUATION
# ───────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("FINAL EVALUATION")
print("=" * 60)

print(f"Best epoch: {best_epoch}")
print(f"Best validation S: {best_val_s:.4f}")

# load best model

model.load_state_dict(
    torch.load(
        f"{LOCAL_OUT}/best_model/pytorch_model.bin",
        map_location=device,
        weights_only=True
    )
)

model.eval()

# ───────────────────────────────────────────────────────────
# COLLECT FUNCTION
# ───────────────────────────────────────────────────────────

def collect_predictions(loader):

    logits_list = []
    labels_list = []
    masks_list  = []

    with torch.no_grad():

        for batch in loader:

            batch = {
                k: v.to(device)
                for k, v in batch.items()
            }

            logits = model(
                batch["input_ids"],
                batch["attention_mask"],
                batch.get("token_type_ids")
            )

            logits_list.append(logits.cpu().numpy())
            labels_list.append(batch["labels"].cpu().numpy())
            masks_list.append(batch["label_mask"].cpu().numpy())

    return (
        np.concatenate(logits_list),
        np.concatenate(labels_list),
        np.concatenate(masks_list),
    )

# ───────────────────────────────────────────────────────────
# THRESHOLD TUNING
# ───────────────────────────────────────────────────────────

val_logits, val_labels, val_masks = collect_predictions(val_loader)

best_thresholds = tune_thresholds(
    val_logits,
    val_labels,
    val_masks,
    metric="s_score"
)

print("\nFINAL THRESHOLDS")
print(best_thresholds)

# ───────────────────────────────────────────────────────────
# TEST EVALUATION
# ───────────────────────────────────────────────────────────

test_logits, test_labels, test_masks = collect_predictions(test_loader)

test_metrics = compute_multilabel_metrics(
    test_logits,
    test_labels,
    test_masks,
    thresholds=best_thresholds,
    split_name="test"
)

print_metrics_table(
    "FINAL TEST METRICS",
    test_metrics
)

# ───────────────────────────────────────────────────────────
# SAVE FINAL FILES
# ───────────────────────────────────────────────────────────

test_metrics.to_csv(
    f"{LOCAL_OUT}/test_metrics.csv",
    index=False
)

with open(f"{LOCAL_OUT}/thresholds.json", "w") as f:

    json.dump(
        best_thresholds,
        f,
        indent=2
    )

summary = {
    "model_name": MODEL_NAME,
    "mode": MODE,
    "max_length": MAX_LEN,
    "batch_size": BATCH,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "warmup_ratio": WARMUP_RATIO,
    "best_epoch": best_epoch,
    "best_val_s": best_val_s,
    "epochs_done": epoch_results,
    "status": "done",
    "test_macro": (
        test_metrics[
            test_metrics["label"] == "MACRO"
        ]
        .iloc[0]
        .to_dict()
    ),
}

with open(f"{LOCAL_OUT}/summary.json", "w") as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

# ───────────────────────────────────────────────────────────
# FINAL SYNC
# ───────────────────────────────────────────────────────────

sync_to_drive()

print("\n✅ TRAINING FINISHED")
print(f"Results saved to:\n{DRIVE_OUT}")

## EXPERIMENT E5
 Model:   bert-base-german-cased

 Data:    gold_silver

 Loss:    MaskedFocalLoss (gamma=2.0, per-label alpha)

 Tokens:  max_length=128

 Batch:   16

 Stop:    Early stopping (patience=2)

 Goal:    Test ob Focal Loss threat F1 > 0 erreicht

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell: bert-base gold_silver 128 + MaskedFocalLoss
#       + Early Stopping + Per-label alpha
# ═══════════════════════════════════════════════════════════
import os, json, random, shutil
import numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

from src.training.transformer_dataset import (
    LABELS, NetiquetteTransformerDataset,
    load_dataset, print_dataset_summary)
from src.training.losses import MaskedFocalLoss
from src.training.transformer_metrics import (
    compute_multilabel_metrics, tune_thresholds, print_metrics_table)

# ── Config ─────────────────────────────────────────────────
MODEL_NAME   = "deepset/gbert-large"
MODE         = "gold_silver"
MAX_LEN      = 128
BATCH        = 4
EPOCHS       = 5
LR           = 1e-5
WARMUP_RATIO = 0.03
GAMMA        = 2.0
PATIENCE     = 2

LOCAL_OUT = "results/gbert_large_gold_silver_128_focal"
DRIVE_OUT = "/content/drive/MyDrive/masterarbeit/results/gbert_large_gold_silver_128_focal"
DATA_PATH    = "data/final/unified_final_v1.parquet"

# ── Pre-flight ─────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError("No GPU!")
device = torch.device("cuda")
print(f" GPU:  {torch.cuda.get_device_name(0)}")
print(f" VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

os.makedirs(f"{LOCAL_OUT}/best_model", exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

# ── Compute per-label alpha từ training data ───────────────
print("\nComputing per-label alpha...")
df_full = __import__("pandas").read_parquet(DATA_PATH)
train_df = df_full[df_full["split"] == "train"].copy()

alpha_values = []
for label in LABELS:
    ann = train_df[label].notna()
    pos = (train_df.loc[ann, label] == 1.0).sum()
    neg = (train_df.loc[ann, label] == 0.0).sum()
    # alpha_pos = neg/(pos+neg)
    alpha_pos = neg / (pos + neg) if (pos + neg) > 0 else 0.5
    alpha_values.append(float(alpha_pos))
    print(f"  {label:<12}: pos={pos:>7,}  neg={neg:>7,}  alpha={alpha_pos:.4f}")

alpha_tensor = torch.tensor(alpha_values, dtype=torch.float32)
print(f"\nAlpha vector: {alpha_values}")

# ── Save config ────────────────────────────────────────────
json.dump({
    "model_name": MODEL_NAME, "mode": MODE,
    "max_length": MAX_LEN,   "batch_size": BATCH,
    "epochs": EPOCHS,        "patience": PATIENCE,
    "learning_rate": LR,     "warmup_ratio": WARMUP_RATIO,
    "loss": "MaskedFocalLoss",
    "gamma": GAMMA,          "alpha": alpha_values,
    "optimizer": "AdamW",    "weight_decay": 0.01,
    "seed": 42,
}, open(f"{LOCAL_OUT}/config.json", "w"), indent=2, ensure_ascii=False)

print(f"\nModel:        {MODEL_NAME}")
print(f"Mode:         {MODE}")
print(f"Max length:   {MAX_LEN}")
print(f"Batch size:   {BATCH}")
print(f"Loss:         MaskedFocalLoss (gamma={GAMMA})")
print(f"Early stop:   patience={PATIENCE}")
print(f"Drive output: {DRIVE_OUT}")

# ── Sync function ──────────────────────────────────────────
def sync_to_drive():
    for fname in ["config.json", "summary_partial.json",
                  "val_metrics_latest.csv", "val_tuned_latest.csv",
                  "thresholds.json", "test_metrics.csv", "summary.json"]:
        src = f"{LOCAL_OUT}/{fname}"
        if os.path.exists(src):
            shutil.copy2(src, f"{DRIVE_OUT}/{fname}")
    for fname in os.listdir(LOCAL_OUT):
        if (fname.startswith("val_metrics_epoch_") or
            fname.startswith("val_tuned_epoch_")) and fname.endswith(".csv"):
            shutil.copy2(f"{LOCAL_OUT}/{fname}", f"{DRIVE_OUT}/{fname}")
    local_best = f"{LOCAL_OUT}/best_model"
    drive_best = f"{DRIVE_OUT}/best_model"
    if os.path.exists(local_best) and os.listdir(local_best):
        os.makedirs(drive_best, exist_ok=True)
        for fname in os.listdir(local_best):
            shutil.copy2(f"{local_best}/{fname}", f"{drive_best}/{fname}")
    print("   Synced to Drive")

# ── Model ──────────────────────────────────────────────────
class TransformerClassifier(nn.Module):
    def __init__(self, model_name, num_labels=4, dropout=0.2):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kw["token_type_ids"] = token_type_ids
        out = self.encoder(**kw)
        pooled = out.pooler_output if (
            hasattr(out, "pooler_output") and out.pooler_output is not None
        ) else out.last_hidden_state[:, 0]
        return self.classifier(self.dropout(pooled))

# ── Setup ──────────────────────────────────────────────────
random.seed(42); np.random.seed(42)
torch.manual_seed(42); torch.cuda.manual_seed_all(42)

splits    = load_dataset(DATA_PATH, mode=MODE)
print_dataset_summary(splits)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def make_loader(df, shuffle):
    ds = NetiquetteTransformerDataset(df, tokenizer, MAX_LEN)
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle,
                      num_workers=0, pin_memory=True)

train_loader = make_loader(splits["train"], True)
val_loader   = make_loader(splits["val"],   False)
test_loader  = make_loader(splits["test"],  False)

model   = TransformerClassifier(MODEL_NAME).to(device)
loss_fn = MaskedFocalLoss(alpha=alpha_tensor, gamma=GAMMA)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(total_steps * WARMUP_RATIO), total_steps)

print(f"\nSteps/epoch: {len(train_loader):,}")
print(f"Total steps: {total_steps:,}")
print(f"Warmup:      {int(total_steps * WARMUP_RATIO):,}")

sync_to_drive()

# ── Training loop mit Early Stopping ──────────────────────
best_val_s    = -1.0
best_epoch    = -1
patience_left = PATIENCE
epoch_results = []

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'='*60}\nEPOCH {epoch}/{EPOCHS}  (patience left: {patience_left})\n{'='*60}")

    # Train
    model.train()
    total_loss, steps = 0.0, 0
    for step, batch in enumerate(train_loader, 1):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        logits = model(batch["input_ids"], batch["attention_mask"],
                       batch.get("token_type_ids"))
        loss = loss_fn(logits, batch["labels"], batch["label_mask"])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += loss.item(); steps += 1
        if step % 500 == 0:
            print(f"  step {step}/{len(train_loader):,} loss={total_loss/steps:.4f}")

    train_loss = total_loss / steps
    print(f"\nEpoch {epoch} train loss: {train_loss:.4f}")

    # Validate
    model.eval()
    all_l, all_lb, all_m = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(batch["input_ids"], batch["attention_mask"],
                           batch.get("token_type_ids"))
            all_l.append(logits.cpu().numpy())
            all_lb.append(batch["labels"].cpu().numpy())
            all_m.append(batch["label_mask"].cpu().numpy())

    val_logits = np.concatenate(all_l)
    val_labels = np.concatenate(all_lb)
    val_masks  = np.concatenate(all_m)

    # threshold=0.5
    val_metrics = compute_multilabel_metrics(
        val_logits, val_labels, val_masks, split_name="val")
    print_metrics_table(f"VAL Epoch {epoch} threshold=0.5", val_metrics)

    # tuned threshold
    tuned = tune_thresholds(val_logits, val_labels, val_masks, metric="s_score")
    print(f"\n  Tuned thresholds: {tuned}")
    val_tuned = compute_multilabel_metrics(
        val_logits, val_labels, val_masks,
        thresholds=tuned, split_name="val_tuned")
    print_metrics_table(f"VAL Epoch {epoch} TUNED", val_tuned)

    macro_s = float(
        val_tuned[val_tuned["label"] == "MACRO"]["s_score"].iloc[0])

    # Save metrics
    val_metrics.to_csv(f"{LOCAL_OUT}/val_metrics_epoch_{epoch}.csv", index=False)
    val_tuned.to_csv(f"{LOCAL_OUT}/val_tuned_epoch_{epoch}.csv",    index=False)
    val_metrics.to_csv(f"{LOCAL_OUT}/val_metrics_latest.csv",        index=False)
    val_tuned.to_csv(f"{LOCAL_OUT}/val_tuned_latest.csv",            index=False)

    epoch_results.append({
        "epoch": epoch, "train_loss": train_loss, "val_macro_s": macro_s})

    # Early stopping logic
    if macro_s > best_val_s:
        best_val_s    = macro_s
        best_epoch    = epoch
        patience_left = PATIENCE   # reset patience
        torch.save(model.state_dict(),
                   f"{LOCAL_OUT}/best_model/pytorch_model.bin")
        tokenizer.save_pretrained(f"{LOCAL_OUT}/best_model")
        print(f"\n  → New best: S={macro_s:.4f} | patience reset to {PATIENCE}")
    else:
        patience_left -= 1
        print(f"\n  → No improvement. patience left: {patience_left}")
        if patience_left <= 0:
            print(f"\n  ⏹ Early stopping triggered at epoch {epoch}!")
            json.dump({
                "model_name": MODEL_NAME, "mode": MODE,
                "epoch": epoch, "best_epoch": best_epoch,
                "best_val_s": best_val_s, "epochs_done": epoch_results,
                "status": "early_stopped",
            }, open(f"{LOCAL_OUT}/summary_partial.json", "w"),
               indent=2, ensure_ascii=False)
            sync_to_drive()
            break

    json.dump({
        "model_name": MODEL_NAME, "mode": MODE,
        "epoch": epoch, "best_epoch": best_epoch,
        "best_val_s": best_val_s, "epochs_done": epoch_results,
        "status": "training",
    }, open(f"{LOCAL_OUT}/summary_partial.json", "w"),
       indent=2, ensure_ascii=False)

    sync_to_drive()

# ── Final evaluation ───────────────────────────────────────
print(f"\nBest epoch: {best_epoch}  |  val S={best_val_s:.4f}")

model.load_state_dict(torch.load(
    f"{LOCAL_OUT}/best_model/pytorch_model.bin",
    map_location=device, weights_only=True))
model.eval()

def collect(loader):
    ll, lb, lm = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(batch["input_ids"], batch["attention_mask"],
                           batch.get("token_type_ids"))
            ll.append(logits.cpu().numpy())
            lb.append(batch["labels"].cpu().numpy())
            lm.append(batch["label_mask"].cpu().numpy())
    return np.concatenate(ll), np.concatenate(lb), np.concatenate(lm)

val_logits, val_labels, val_masks = collect(val_loader)
best_thresholds = tune_thresholds(
    val_logits, val_labels, val_masks, metric="s_score")
print("Final thresholds:", best_thresholds)

test_logits, test_labels, test_masks = collect(test_loader)
test_metrics = compute_multilabel_metrics(
    test_logits, test_labels, test_masks,
    thresholds=best_thresholds, split_name="test")
print_metrics_table("FINAL TEST METRICS", test_metrics)

test_metrics.to_csv(f"{LOCAL_OUT}/test_metrics.csv", index=False)
json.dump(best_thresholds,
          open(f"{LOCAL_OUT}/thresholds.json", "w"), indent=2)
json.dump({
    "model_name": MODEL_NAME, "mode": MODE,
    "max_length": MAX_LEN,   "batch_size": BATCH,
    "loss": "MaskedFocalLoss", "gamma": GAMMA, "alpha": alpha_values,
    "best_epoch": best_epoch,"best_val_s": best_val_s,
    "epochs_done": epoch_results, "status": "done",
    "test_macro": test_metrics[
        test_metrics["label"] == "MACRO"].iloc[0].to_dict(),
}, open(f"{LOCAL_OUT}/summary.json", "w"),
   indent=2, ensure_ascii=False)

sync_to_drive()
print(f"\n Done! Results at:\n  {DRIVE_OUT}")

## EXPERIMENT E7  
MODEL_NAME   = "deepset/gbert-large"

MODE         = "gold_silver"       # ←
CHANGED

MAX_LEN      = 128

BATCH        = 8

EPOCHS       = 5

LR           = 5e-6                # ←
CHANGED: kleiner als gold_only

WARMUP_RATIO = 0.10                # ←
CHANGED: längeres Warmup

PATIENCE     = 2

GAMMA        = 2.0

LOCAL_OUT    = "results/gbert_large_gold_silver_128_focal"

DRIVE_OUT    = "/content/drive/MyDrive/masterarbeit/results/gbert_large_gold_silver_128_focal_lr5e6"

DATA_PATH    = "data/final/unified_final_v1.parquet"

```
Cell: gbert large gold_silver 128 + MaskedFocalLoss + Early Stopping + Per-label alpha

```



In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell: gbert-large gold_silver 128 + MaskedFocalLoss
#       LR=5e-6, batch=8, warmup=0.10
#       + TRUE RESUME SUPPORT during epochs
# ═══════════════════════════════════════════════════════════

import os, json, random, shutil
import numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

from src.training.transformer_dataset import (
    LABELS, NetiquetteTransformerDataset,
    load_dataset, print_dataset_summary
)
from src.training.losses import MaskedFocalLoss
from src.training.transformer_metrics import (
    compute_multilabel_metrics, tune_thresholds, print_metrics_table
)

# ── Config ─────────────────────────────────────────────────
MODEL_NAME   = "deepset/gbert-large"
MODE         = "gold_silver"
MAX_LEN      = 128
BATCH        = 8
EPOCHS       = 5
LR           = 5e-6
WARMUP_RATIO = 0.10
PATIENCE     = 2
GAMMA        = 2.0

LOCAL_OUT = "results/gbert_large_gold_silver_128_focal_lr5e6"
DRIVE_OUT = "/content/drive/MyDrive/masterarbeit/results/gbert_large_gold_silver_128_focal_lr5e6"
DATA_PATH = "data/final/unified_final_v1.parquet"

CHECKPOINT_LOCAL = f"{LOCAL_OUT}/checkpoint_latest.pt"
CHECKPOINT_DRIVE = f"{DRIVE_OUT}/checkpoint_latest.pt"

SAVE_EVERY_STEPS = 5000

# ── Pre-flight ─────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError("No GPU!")

device = torch.device("cuda")
print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

os.makedirs(f"{LOCAL_OUT}/best_model", exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

# ── Restore checkpoint from Drive ──────────────────────────
if os.path.exists(CHECKPOINT_DRIVE):
    shutil.copy2(CHECKPOINT_DRIVE, CHECKPOINT_LOCAL)
    print(f"\nResume checkpoint found on Drive: {CHECKPOINT_DRIVE}")
else:
    print("\nNo step checkpoint found. Starting fresh or epoch-resume only.")

# ── Alpha ──────────────────────────────────────────────────
print("\nComputing per-label alpha...")
df_full = __import__("pandas").read_parquet(DATA_PATH)
train_df = df_full[df_full["split"] == "train"].copy()

alpha_values = []
for label in LABELS:
    ann = train_df[label].notna()
    pos = (train_df.loc[ann, label] == 1.0).sum()
    neg = (train_df.loc[ann, label] == 0.0).sum()
    alpha_pos = neg / (pos + neg) if (pos + neg) > 0 else 0.5
    alpha_values.append(float(alpha_pos))
    print(f"  {label:<12}: pos={pos:>7,} neg={neg:>7,} alpha={alpha_pos:.4f}")

alpha_tensor = torch.tensor(alpha_values, dtype=torch.float32)

# ── Save config ────────────────────────────────────────────
json.dump({
    "model_name": MODEL_NAME,
    "mode": MODE,
    "max_length": MAX_LEN,
    "batch_size": BATCH,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "learning_rate": LR,
    "warmup_ratio": WARMUP_RATIO,
    "loss": "MaskedFocalLoss",
    "gamma": GAMMA,
    "alpha": alpha_values,
    "optimizer": "AdamW",
    "weight_decay": 0.01,
    "seed": 42,
    "resume_support": "step_level",
    "save_every_steps": SAVE_EVERY_STEPS,
}, open(f"{LOCAL_OUT}/config.json", "w"), indent=2, ensure_ascii=False)

# ── Sync function ──────────────────────────────────────────
def sync_to_drive():
    for fname in [
        "config.json", "summary_partial.json",
        "val_metrics_latest.csv", "val_tuned_latest.csv",
        "thresholds.json", "test_metrics.csv", "summary.json"
    ]:
        src = f"{LOCAL_OUT}/{fname}"
        if os.path.exists(src):
            shutil.copy2(src, f"{DRIVE_OUT}/{fname}")

    for fname in os.listdir(LOCAL_OUT):
        if (
            fname.startswith("val_metrics_epoch_")
            or fname.startswith("val_tuned_epoch_")
        ) and fname.endswith(".csv"):
            shutil.copy2(f"{LOCAL_OUT}/{fname}", f"{DRIVE_OUT}/{fname}")

    local_best = f"{LOCAL_OUT}/best_model"
    drive_best = f"{DRIVE_OUT}/best_model"
    if os.path.exists(local_best) and os.listdir(local_best):
        os.makedirs(drive_best, exist_ok=True)
        for fname in os.listdir(local_best):
            shutil.copy2(f"{local_best}/{fname}", f"{drive_best}/{fname}")

    if os.path.exists(CHECKPOINT_LOCAL):
        shutil.copy2(CHECKPOINT_LOCAL, CHECKPOINT_DRIVE)

    print("Synced to Drive")

# ── Model ──────────────────────────────────────────────────
class TransformerClassifier(nn.Module):
    def __init__(self, model_name, num_labels=4, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(
            self.encoder.config.hidden_size,
            num_labels
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        }
        if token_type_ids is not None:
            kw["token_type_ids"] = token_type_ids

        out = self.encoder(**kw)
        pooled = (
            out.pooler_output
            if hasattr(out, "pooler_output") and out.pooler_output is not None
            else out.last_hidden_state[:, 0]
        )
        return self.classifier(self.dropout(pooled))

# ── Setup ──────────────────────────────────────────────────
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

splits = load_dataset(DATA_PATH, mode=MODE)
print_dataset_summary(splits)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def make_loader(df, shuffle):
    ds = NetiquetteTransformerDataset(df, tokenizer, MAX_LEN)
    return DataLoader(
        ds,
        batch_size=BATCH,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=True,
    )

train_loader = make_loader(splits["train"], True)
val_loader   = make_loader(splits["val"], False)
test_loader  = make_loader(splits["test"], False)

model = TransformerClassifier(MODEL_NAME).to(device)
loss_fn = MaskedFocalLoss(alpha=alpha_tensor, gamma=GAMMA)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.01
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    warmup_steps,
    total_steps
)

# ── Resume state defaults ──────────────────────────────────
start_epoch = 1
resume_step = 0

best_val_s = -1.0
best_epoch = -1
patience_left = PATIENCE
epoch_results = []

# ── Load step checkpoint if exists ─────────────────────────
if os.path.exists(CHECKPOINT_LOCAL):
    print(f"\nLoading checkpoint: {CHECKPOINT_LOCAL}")
    ckpt = torch.load(CHECKPOINT_LOCAL, map_location=device)

    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])

    start_epoch = ckpt["epoch"]
    resume_step = ckpt["step"]

    best_val_s = ckpt.get("best_val_s", -1.0)
    best_epoch = ckpt.get("best_epoch", -1)
    patience_left = ckpt.get("patience_left", PATIENCE)
    epoch_results = ckpt.get("epoch_results", [])

    print(f"Resuming from epoch={start_epoch}, step={resume_step}")
    print(f"Best so far: S={best_val_s:.4f} at epoch {best_epoch}")
    print(f"Patience left: {patience_left}")
else:
    print("\nStarting training from scratch.")

print(f"\nModel:        {MODEL_NAME}")
print(f"Mode:         {MODE}")
print(f"Max length:   {MAX_LEN}")
print(f"Batch size:   {BATCH}")
print(f"LR:           {LR}")
print(f"Warmup ratio: {WARMUP_RATIO}")
print(f"Loss:         MaskedFocalLoss gamma={GAMMA}")
print(f"Steps/epoch:  {len(train_loader):,}")
print(f"Total steps:  {total_steps:,}")
print(f"Warmup steps: {warmup_steps:,}")
print(f"Drive output: {DRIVE_OUT}")

sync_to_drive()

# ── Checkpoint save ────────────────────────────────────────
def save_checkpoint(epoch, step):
    ckpt = {
        "epoch": epoch,
        "step": step,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_s": best_val_s,
        "best_epoch": best_epoch,
        "patience_left": patience_left,
        "epoch_results": epoch_results,
    }
    torch.save(ckpt, CHECKPOINT_LOCAL)
    shutil.copy2(CHECKPOINT_LOCAL, CHECKPOINT_DRIVE)
    print(f"Checkpoint saved: epoch={epoch}, step={step}")

# ── Training loop ──────────────────────────────────────────
for epoch in range(start_epoch, EPOCHS + 1):

    if patience_left <= 0:
        print(f"\nEarly stopping before epoch {epoch}")
        break

    print(f"\n{'=' * 60}")
    print(f"EPOCH {epoch}/{EPOCHS} | resume_step={resume_step} | patience={patience_left}")
    print(f"{'=' * 60}")

    model.train()
    total_loss = 0.0
    steps = 0

    for step, batch in enumerate(train_loader, 1):

        # Skip already completed batches in current epoch
        if epoch == start_epoch and step <= resume_step:
            continue

        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        logits = model(
            batch["input_ids"],
            batch["attention_mask"],
            batch.get("token_type_ids")
        )

        loss = loss_fn(logits, batch["labels"], batch["label_mask"])
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        steps += 1

        if step % 500 == 0:
            avg_loss = total_loss / max(steps, 1)
            print(f"  step {step}/{len(train_loader):,} loss={avg_loss:.4f}")

        if step % SAVE_EVERY_STEPS == 0:
            save_checkpoint(epoch, step)

    if steps == 0:
        print(f"No new steps trained in epoch {epoch}. Skipping validation.")
        resume_step = 0
        continue

    train_loss = total_loss / steps
    print(f"\nEpoch {epoch} train loss: {train_loss:.4f}")

    # ── Validation ─────────────────────────────────────────
    model.eval()
    all_l, all_lb, all_m = [], [], []

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            logits = model(
                batch["input_ids"],
                batch["attention_mask"],
                batch.get("token_type_ids")
            )

            all_l.append(logits.cpu().numpy())
            all_lb.append(batch["labels"].cpu().numpy())
            all_m.append(batch["label_mask"].cpu().numpy())

    val_logits = np.concatenate(all_l)
    val_labels = np.concatenate(all_lb)
    val_masks  = np.concatenate(all_m)

    val_metrics = compute_multilabel_metrics(
        val_logits,
        val_labels,
        val_masks,
        split_name="val"
    )
    print_metrics_table(f"VAL Epoch {epoch} threshold=0.5", val_metrics)

    tuned = tune_thresholds(
        val_logits,
        val_labels,
        val_masks,
        metric="s_score"
    )
    print(f"\nTuned thresholds: {tuned}")

    val_tuned = compute_multilabel_metrics(
        val_logits,
        val_labels,
        val_masks,
        thresholds=tuned,
        split_name="val_tuned"
    )
    print_metrics_table(f"VAL Epoch {epoch} TUNED", val_tuned)

    macro_s = float(
        val_tuned[val_tuned["label"] == "MACRO"]["s_score"].iloc[0]
    )

    val_metrics.to_csv(
        f"{LOCAL_OUT}/val_metrics_epoch_{epoch}.csv",
        index=False
    )
    val_tuned.to_csv(
        f"{LOCAL_OUT}/val_tuned_epoch_{epoch}.csv",
        index=False
    )
    val_metrics.to_csv(
        f"{LOCAL_OUT}/val_metrics_latest.csv",
        index=False
    )
    val_tuned.to_csv(
        f"{LOCAL_OUT}/val_tuned_latest.csv",
        index=False
    )

    epoch_results.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_macro_s": macro_s,
    })

    if macro_s > best_val_s:
        best_val_s = macro_s
        best_epoch = epoch
        patience_left = PATIENCE

        torch.save(
            model.state_dict(),
            f"{LOCAL_OUT}/best_model/pytorch_model.bin"
        )
        tokenizer.save_pretrained(f"{LOCAL_OUT}/best_model")

        print(f"\nNew best: S={macro_s:.4f} | patience reset to {PATIENCE}")

    else:
        patience_left -= 1
        print(f"\nNo improvement. patience left: {patience_left}")

    json.dump({
        "model_name": MODEL_NAME,
        "mode": MODE,
        "epoch": epoch,
        "best_epoch": best_epoch,
        "best_val_s": best_val_s,
        "epochs_done": epoch_results,
        "status": "training" if patience_left > 0 else "early_stopped",
    }, open(f"{LOCAL_OUT}/summary_partial.json", "w"),
       indent=2, ensure_ascii=False)

    # Save checkpoint for next epoch start
    resume_step = 0
    save_checkpoint(epoch + 1, 0)
    sync_to_drive()

    if patience_left <= 0:
        print(f"\nEarly stopping triggered at epoch {epoch}")
        break

# ── Final evaluation ───────────────────────────────────────
print(f"\nBest epoch: {best_epoch} | val S={best_val_s:.4f}")

best_model_path = f"{LOCAL_OUT}/best_model/pytorch_model.bin"
if not os.path.exists(best_model_path):
    raise FileNotFoundError("No best model found. Training may not have completed one epoch.")

model.load_state_dict(
    torch.load(best_model_path, map_location=device, weights_only=True)
)
model.eval()

def collect(loader):
    ll, lb, lm = [], [], []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            logits = model(
                batch["input_ids"],
                batch["attention_mask"],
                batch.get("token_type_ids")
            )

            ll.append(logits.cpu().numpy())
            lb.append(batch["labels"].cpu().numpy())
            lm.append(batch["label_mask"].cpu().numpy())

    return (
        np.concatenate(ll),
        np.concatenate(lb),
        np.concatenate(lm),
    )

val_logits, val_labels, val_masks = collect(val_loader)
best_thresholds = tune_thresholds(
    val_logits,
    val_labels,
    val_masks,
    metric="s_score"
)

print("Final thresholds:", best_thresholds)

test_logits, test_labels, test_masks = collect(test_loader)
test_metrics = compute_multilabel_metrics(
    test_logits,
    test_labels,
    test_masks,
    thresholds=best_thresholds,
    split_name="test"
)

print_metrics_table("FINAL TEST METRICS", test_metrics)

test_metrics.to_csv(f"{LOCAL_OUT}/test_metrics.csv", index=False)

np.savez(
    f"{LOCAL_OUT}/test_logits.npz",
    logits=test_logits,
    labels=test_labels,
    label_mask=test_masks,
)

np.savez(
    f"{LOCAL_OUT}/val_logits.npz",
    logits=val_logits,
    labels=val_labels,
    label_mask=val_masks,
)

json.dump(
    best_thresholds,
    open(f"{LOCAL_OUT}/thresholds.json", "w"),
    indent=2
)

json.dump({
    "model_name": MODEL_NAME,
    "mode": MODE,
    "max_length": MAX_LEN,
    "batch_size": BATCH,
    "learning_rate": LR,
    "warmup_ratio": WARMUP_RATIO,
    "loss": "MaskedFocalLoss",
    "gamma": GAMMA,
    "alpha": alpha_values,
    "best_epoch": best_epoch,
    "best_val_s": best_val_s,
    "epochs_done": epoch_results,
    "status": "done",
    "test_macro": test_metrics[
        test_metrics["label"] == "MACRO"
    ].iloc[0].to_dict(),
}, open(f"{LOCAL_OUT}/summary.json", "w"),
   indent=2, ensure_ascii=False)

sync_to_drive()

print(f"\nDone! Results at:\n  {DRIVE_OUT}")

GPU:  Tesla T4
VRAM: 15.6 GB

Resume checkpoint found on Drive: /content/drive/MyDrive/masterarbeit/results/gbert_large_gold_silver_128_focal_lr5e6/checkpoint_latest.pt

Computing per-label alpha...
  hate_speech : pos=  8,637 neg=269,878 alpha=0.9690
  toxic       : pos= 34,936 neg=305,831 alpha=0.8975
  threat      : pos=    735 neg=235,964 alpha=0.9969
  insult      : pos= 19,372 neg=266,942 alpha=0.9323

Dataset summary:

TRAIN:
  rows   : 426,742
  gold   : 61,830
  silver : 364,912
  hate_speech  annotated=278,515 pos= 8,637 neg=269,878
  toxic        annotated=340,767 pos=34,936 neg=305,831
  threat       annotated=236,699 pos=   735 neg=235,964
  insult       annotated=286,314 pos=19,372 neg=266,942

VAL:
  rows   : 13,250
  gold   : 13,250
  silver : 0
  hate_speech  annotated= 13,250 pos= 1,422 neg= 11,828
  toxic        annotated=  2,722 pos=   819 neg=  1,903
  threat       annotated=  4,295 pos=    21 neg=  4,274
  insult       annotated=  7,017 pos= 1,296 neg=  5,721

TES

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: deepset/gbert-large
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Loading checkpoint: results/gbert_large_gold_silver_128_focal_lr5e6/checkpoint_latest.pt
Resuming from epoch=5, step=35000
Best so far: S=0.6477 at epoch 4
Patience left: 2

Model:        deepset/gbert-large
Mode:         gold_silver
Max length:   128
Batch size:   8
LR:           5e-06
Warmup ratio: 0.1
Loss:         MaskedFocalLoss gamma=2.0
Steps/epoch:  53,343
Total steps:  266,715
Warmup steps: 26,671
Drive output: /content/drive/MyDrive/masterarbeit/results/gbert_large_gold_silver_128_focal_lr5e6
Synced to Drive

EPOCH 5/5 | resume_step=35000 | patience=2
  step 35500/53,343 loss=0.0031
  step 36000/53,343 loss=0.0027
  step 36500/53,343 loss=0.0027
  step 37000/53,343 loss=0.0025
  step 37500/53,343 loss=0.0025
  step 38000/53,343 loss=0.0025
  step 38500/53,343 loss=0.0025
  step 39000/53,343 loss=0.0025
  step 39500/53,343 loss=0.0024
  step 40000/53,343 loss=0.0025
Checkpoint saved: epoch=5, step=40000
  step 40500/53,343 loss=0.0024
  step 41000/53,343 loss=0.0024
  step 41

FileNotFoundError: No best model found. Training may not have completed one epoch.

In [ ]:
# ── Fix: Copy best model từ Drive về local ─────────────────
import os, shutil, json, torch
import numpy as np

LOCAL_OUT = "results/gbert_large_gold_silver_128_focal_lr5e6"
DRIVE_OUT = "/content/drive/MyDrive/masterarbeit/results/gbert_large_gold_silver_128_focal_lr5e6"

# Copy best_model từ Drive
os.makedirs(f"{LOCAL_OUT}/best_model", exist_ok=True)
drive_best = f"{DRIVE_OUT}/best_model"
for fname in os.listdir(drive_best):
    shutil.copy2(f"{drive_best}/{fname}", f"{LOCAL_OUT}/best_model/{fname}")
    print(f"Copied: {fname}")

print("✅ Best model restored!")

# ── Chạy final evaluation ───────────────────────────────────
from src.training.transformer_dataset import LABELS, NetiquetteTransformerDataset, load_dataset
from src.training.transformer_metrics import compute_multilabel_metrics, tune_thresholds, print_metrics_table
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer
import torch.nn as nn

MODEL_NAME = "deepset/gbert-large"
MAX_LEN    = 128
BATCH      = 8
DATA_PATH  = "data/final/unified_final_v1.parquet"
device     = torch.device("cuda")

class TransformerClassifier(nn.Module):
    def __init__(self, model_name, num_labels=4, dropout=0.2):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kw["token_type_ids"] = token_type_ids
        out = self.encoder(**kw)
        pooled = out.pooler_output if (
            hasattr(out, "pooler_output") and out.pooler_output is not None
        ) else out.last_hidden_state[:, 0]
        return self.classifier(self.dropout(pooled))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
splits    = load_dataset(DATA_PATH, mode="gold_silver")

def make_loader(df, shuffle):
    ds = NetiquetteTransformerDataset(df, tokenizer, MAX_LEN)
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle, num_workers=0, pin_memory=True)

val_loader  = make_loader(splits["val"],  False)
test_loader = make_loader(splits["test"], False)

model = TransformerClassifier(MODEL_NAME).to(device)
model.load_state_dict(torch.load(
    f"{LOCAL_OUT}/best_model/pytorch_model.bin",
    map_location=device, weights_only=True))
model.eval()

def collect(loader):
    ll, lb, lm = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(batch["input_ids"], batch["attention_mask"],
                           batch.get("token_type_ids"))
            ll.append(logits.cpu().numpy())
            lb.append(batch["labels"].cpu().numpy())
            lm.append(batch["label_mask"].cpu().numpy())
    return np.concatenate(ll), np.concatenate(lb), np.concatenate(lm)

val_logits, val_labels, val_masks = collect(val_loader)
best_thresholds = tune_thresholds(val_logits, val_labels, val_masks, metric="s_score")
print("Final thresholds:", best_thresholds)

test_logits, test_labels, test_masks = collect(test_loader)
test_metrics = compute_multilabel_metrics(
    test_logits, test_labels, test_masks,
    thresholds=best_thresholds, split_name="test")
print_metrics_table("FINAL TEST METRICS", test_metrics)

import pandas as pd
test_metrics.to_csv(f"{LOCAL_OUT}/test_metrics.csv", index=False)
json.dump(best_thresholds, open(f"{LOCAL_OUT}/thresholds.json", "w"), indent=2)

# Sync to Drive
import shutil
for fname in ["test_metrics.csv", "thresholds.json"]:
    shutil.copy2(f"{LOCAL_OUT}/{fname}", f"{DRIVE_OUT}/{fname}")
print(" Done! Test metrics saved.")

Copied: pytorch_model.bin
Copied: tokenizer_config.json
Copied: tokenizer.json
✅ Best model restored!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: deepset/gbert-large
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Final thresholds: {'hate_speech': 0.55, 'toxic': 0.3, 'threat': 0.8, 'insult': 0.45}

FINAL TEST METRICS
----------------------------------------------------------------------------------------------------
      label  threshold  precision  recall     f1     f2    mcc  s_score  support_pos  support_total
hate_speech     0.5500     0.3227  0.7152 0.4447 0.5752 0.3887   0.6348         1422          13250
      toxic     0.3000     0.5984  0.8315 0.6960 0.7714 0.5499   0.7732          819           2722
     threat     0.8000     0.2500  0.3333 0.2857 0.3125 0.2847   0.4774           21           4376
     insult     0.4500     0.4258  0.7616 0.5462 0.6578 0.4384   0.6885         1296           7098
      MACRO        NaN     0.3992  0.6604 0.4931 0.5792 0.4154   0.6435         3558          27446
✅ Done! Test metrics saved.


# Experiment E8 — gbert-large + MaskedASL (Asymmetric Loss)

## Übersicht

| Parameter | Wert |
|---|---|
| **Experiment** | E8 |
| **Modell** | `deepset/gbert-large` |
| **Datenmodus** | `gold_silver` |
| **Verlustfunktion** | MaskedASL |
| **Lernrate** | 2e-5 |
| **Batch-Größe** | 8 |
| **Max. Sequenzlänge** | 128 |
| **Epochs (max)** | 5 |
| **Early Stopping (Patience)** | 2 |
| **Referenz** | Ridnik et al. (2021, ICCV) — arXiv:2009.14119 |

---

## Motivation

E7 (gbert-large + Focal Loss + gold_silver) erzielte Macro-S = 0.644 und ist das bisher beste BERT-Experiment. Die Hauptschwäche liegt beim Label `threat`, das mit nur **21 positiven Beispielen** im Testset (von 4.376 annotierten Samples) extrem selten ist:

- E7 threat-F1 = 0.286
- E7 threat-Recall = 0.333

Focal Loss behandelt alle Labels gleich. Die **Asymmetric Loss (ASL)** adressiert dieses Problem durch zwei Mechanismen:

1. **Asymmetrisches Gamma**: Negative Beispiele werden stärker down-gewichtet als positive → das Modell fokussiert sich mehr auf rare positive Samples.
2. **Hard Thresholding (Clip)**: Sehr sichere Negative (p < 0.05) tragen keinen Gradient → verhindert, dass noisy Silver-Label-Annotationen das Training stören.

---

## ASL-Parameter

```
gamma_pos = 0      → kein Down-weighting für positive Samples
gamma_neg = 6      → stärkeres Down-weighting als Focal (γ=2) oder ASL-default (γ=4)
clip      = 0.05   → Negative mit p < 0.05 → kein Gradient (noisy silver labels)
```

### Per-Label Weighting

Zusätzlich zur ASL werden **label-spezifische Gewichte** eingesetzt:

| Label | Weight | Begründung |
|---|---|---|
| hate_speech | 1.0 | ausreichend annotiert |
| toxic | 1.0 | ausreichend annotiert |
| **threat** | **5.0** | extrem selten (21 pos / 4376 total) |
| insult | 1.0 | ausreichend annotiert |

---

## Verlustfunktion — Formelübersicht

**Positive Seite:**

$$L_{pos} = -(1-p)^{\gamma_{pos}} \cdot \log(p)$$

Mit $\gamma_{pos} = 0$ vereinfacht sich dies zu standardmäßigem Binary Cross-Entropy für Positive:

$$L_{pos} = -\log(p)$$

**Negative Seite mit Hard Threshold:**

$$p_{clip} = \max(p - \delta, 0), \quad \delta = 0.05$$

$$L_{neg} = -(p_{clip})^{\gamma_{neg}} \cdot \log(1 - p_{clip})$$

**Kombiniert mit per-label Gewichten und NaN-Masking:**

$$\mathcal{L}_{ASL} = \frac{\sum_{i,j} m_{ij} \cdot w_j \cdot [y_{ij} \cdot L_{pos}(p_{ij}) + (1 - y_{ij}) \cdot L_{neg}(p_{ij})]}{\sum_{i,j} m_{ij}}$$

Wobei $m_{ij} = 0$ für NaN-annotierte Samples (kein Gradient).

---

## Unterschied zu E7 (Focal Loss)

| Aspekt | E7 Focal Loss | E8 MaskedASL |
|---|---|---|
| Verlustfunktion | Focal Loss (γ=2) | Asymmetric Loss |
| Positive down-weighting | ja (γ=2) | nein (γ_pos=0) |
| Negative down-weighting | ja (γ=2) | stark (γ_neg=6) |
| Noisy label handling | nein | ja (clip=0.05) |
| Per-label weighting | nein | ja (threat=5.0) |
| Lernrate | 2e-5 | 2e-5 |

---

## Erwartete Verbesserung

- **Primäres Ziel:** `threat` S-Score > 0.477 (E7-Baseline)
- **Sekundäres Ziel:** Macro S-Score > 0.644 (E7-Baseline)
- **Mechanismus:** Höheres `gamma_neg` + threat-weight=5.0 → Modell lernt, seltene positive threat-Samples stärker zu gewichten

---

## Verzeichnisstruktur

```
results/gbert_large_gold_silver_128_asl_v2/
├── config.json               Experiment-Konfiguration
├── best_model/               Bestes Modell (nach val S-Score)
│   ├── pytorch_model.bin
│   └── tokenizer files
├── val_metrics_epoch_N.csv   Validierungsmetriken pro Epoch
├── val_tuned_epoch_N.csv     Validierungsmetriken mit Threshold-Tuning
├── thresholds.json           Optimale Thresholds (S-Score tuned)
├── test_metrics.csv          Finale Testmetriken
├── test_logits.npz           Logits für Hybrid-Evaluation (E11)
├── val_logits.npz            Logits für Threshold-Tuning
├── summary.json              Vollständige Zusammenfassung
└── checkpoint_latest.pt      Resume-Checkpoint
```

---

## Verknüpfung mit Forschungsfragen

- **RQ1:** E8 ist Teil der BERT-Modellreihe (E1–E8) zur Beantwortung der Klassifikationsleistung moderner Sprachmodelle.
- **RQ3:** E8 wird direkt mit E9 (qwen2.5:7b) und E10 (Gemini 2.5 Flash) verglichen → Vergleich BERT vs. LLM.
- **RQ4:** Die `test_logits.npz` von E8 werden als BERT-Komponente im hybriden System (E11) verwendet.



In [ ]:
!pip install sentencepiece -q
import importlib, sentencepiece
print("sentencepiece version:", sentencepiece.__version__)

sentencepiece version: 0.2.1


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# E8: gbert-large + gold_silver + MaskedASL (Asymmetric Loss)
#
# Unterschied zu E7 (Focal Loss): nur die Verlustfunktion.
# Alle übrigen Hyperparameter (LR, Batch, MaxLen, Warmup) sind identisch,
# um einen sauberen Ablationsvergleich E7 vs. E8 zu ermöglichen.
#
# ASL-Parameter entsprechen den Standardwerten aus Ben-Baruch et al. (2021):
#   gamma_pos = 0, gamma_neg = 4, clip (m) = 0.05
#
# WARNUNG — Vorheriger Fehlversuch:
#   Ein Lauf mit gamma_neg=6 + label_weights=[1,1,5,1] führte zum Modellkollaps
#   (Recall=1.0, MCC=0.0 für alle Labels). Grund: Bei p≈0.5 beträgt das
#   Gradientengewicht der negativen Seite (0.5-0.05)^6 ≈ 0.008 gegenüber 1.0
#   auf der positiven Seite. Das Modell lernt, konstant 1 vorherzusagen.
#   → gamma_neg = 4 ist die empirische Obergrenze.
#
# Referenz: Ben-Baruch et al. (2021, ICCV) — arXiv:2009.14119
# ═══════════════════════════════════════════════════════════════════════════

import subprocess
subprocess.run(["pip", "install", "sentencepiece", "-q"], check=True)

import os, json, random, shutil
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import BertModel, BertTokenizer, get_linear_schedule_with_warmup

from src.training.transformer_dataset import (
    LABELS, NetiquetteTransformerDataset,
    load_dataset, print_dataset_summary,
)
from src.training.transformer_metrics import (
    compute_multilabel_metrics, tune_thresholds, print_metrics_table,
)

# ── Experiment identity ────────────────────────────────────────────────────
EXP_NAME = "E8"
EXP_NOTE = "gbert-large gold_silver MaskedASL (ASL-Defaults: gamma_neg=4, clip=0.05)"

# ── Config — identisch zu E7 außer Loss ────────────────────────────────────
MODEL_NAME   = "deepset/gbert-large"
MODE         = "gold_silver"
MAX_LEN      = 128
BATCH        = 8
EPOCHS       = 5
LR           = 5e-6        # wie E7
WARMUP_RATIO = 0.10        # wie E7
PATIENCE     = 2

# ── ASL Parameters (Ben-Baruch et al. 2021, Defaults) ──────────────────────
# gamma_pos = 0  → kein Down-weighting positiver Beispiele (seltene Labels)
# gamma_neg = 4  → Down-weighting leichter negativer Beispiele
# clip      = 0.05 → sehr sichere Negative (p<0.05) tragen keinen Gradienten;
#                    robust gegenüber Label-Rauschen in den Silver-Daten
ASL_GAMMA_POS = 0
ASL_GAMMA_NEG = 4
ASL_CLIP      = 0.05

# Keine label-spezifische Gewichtung — ASL adressiert Imbalance bereits
# über die asymmetrischen Gamma-Werte. Zusätzliche Gewichte destabilisieren.
LABEL_WEIGHTS = [1.0, 1.0, 1.0, 1.0]

# ── Paths ──────────────────────────────────────────────────────────────────
RUN_ID           = "gbert_large_gold_silver_128_asl_v3"
LOCAL_OUT        = f"results/{RUN_ID}"
DRIVE_OUT        = f"/content/drive/MyDrive/masterarbeit/results/{RUN_ID}"
DATA_PATH        = "data/final/unified_final_v1.parquet"
CHECKPOINT_LOCAL = f"{LOCAL_OUT}/checkpoint_latest.pt"
CHECKPOINT_DRIVE = f"{DRIVE_OUT}/checkpoint_latest.pt"
SAVE_EVERY_STEPS = 5000

# ── E7 Baseline (für Vergleichstabelle am Ende) ────────────────────────────
E7_MACRO_S  = 0.6440
E7_MACRO_F1 = 0.4930
E7_MACRO_F2 = 0.5790
E7_THREAT_S  = 0.4770
E7_THREAT_F1 = 0.2860

# ── Pre-flight ─────────────────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU available. Colab: Runtime → Change runtime type → T4 GPU"
    )

device = torch.device("cuda")
print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

os.makedirs(f"{LOCAL_OUT}/best_model", exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

# ── Restore checkpoint from Drive ─────────────────────────────────────────
if os.path.exists(CHECKPOINT_DRIVE):
    shutil.copy2(CHECKPOINT_DRIVE, CHECKPOINT_LOCAL)
    print(f"Resume checkpoint found: {CHECKPOINT_DRIVE}")
else:
    print("No checkpoint found. Starting fresh.")


# ── Loss Function ──────────────────────────────────────────────────────────
class MaskedASL(nn.Module):
    """
    Masked Asymmetric Loss (Ben-Baruch et al., 2021).

    Positive Seite:
        L_pos = -(1-p)^gamma_pos * log(p)
        gamma_pos=0 → volles Gewicht für seltene Positiva (kein Down-weighting)

    Negative Seite mit Probability Shifting:
        p_m   = max(p - m, 0)              mit m = clip
        L_neg = -(p_m)^gamma_neg * log(1 - p_m)
        → Negative mit p < m tragen exakt 0 zum Gradienten bei

    Masking:
        Nicht annotierte Labels (NaN) tragen keinen Gradienten (mask=0).

    Hinweis zur Parameterwahl:
        gamma_neg > 4 führte in Vorversuchen zum Modellkollaps, da das
        Gradientengewicht der negativen Seite bei p≈0.5 unter 1% des
        positiven Gewichts fällt.
    """

    def __init__(self, gamma_pos=0, gamma_neg=4, clip=0.05, label_weights=None):
        super().__init__()
        self.gamma_pos = gamma_pos
        self.gamma_neg = gamma_neg
        self.clip = clip

        if label_weights is not None:
            w = torch.tensor(label_weights, dtype=torch.float32)
            self.register_buffer("label_weights", w)
        else:
            self.label_weights = None

    def forward(self, logits, labels, mask):
        p = torch.sigmoid(logits)

        # Positive Seite
        loss_pos = -torch.pow(1.0 - p, self.gamma_pos) * \
                    torch.log(p.clamp(min=1e-8))

        # Negative Seite mit Probability Shifting
        p_m = (p - self.clip).clamp(min=0.0)
        loss_neg = -torch.pow(p_m, self.gamma_neg) * \
                    torch.log((1.0 - p_m).clamp(min=1e-8))

        loss = labels * loss_pos + (1.0 - labels) * loss_neg

        if self.label_weights is not None:
            loss = loss * self.label_weights.unsqueeze(0)

        loss = loss * mask
        return loss.sum() / mask.sum().clamp(min=1.0)


loss_fn = MaskedASL(
    gamma_pos=ASL_GAMMA_POS,
    gamma_neg=ASL_GAMMA_NEG,
    clip=ASL_CLIP,
    label_weights=LABEL_WEIGHTS,
).to(device)

print(f"\nLoss: MaskedASL")
print(f"  gamma_pos     = {ASL_GAMMA_POS}")
print(f"  gamma_neg     = {ASL_GAMMA_NEG}")
print(f"  clip          = {ASL_CLIP}")
print(f"  label_weights = {dict(zip(LABELS, LABEL_WEIGHTS))}")

# Sanity check: relatives Gradientengewicht bei p=0.5
_p = 0.5
_w_neg = (_p - ASL_CLIP) ** ASL_GAMMA_NEG
_w_pos = (1.0 - _p) ** ASL_GAMMA_POS
print(f"  Gradient weight @ p=0.5:  pos={_w_pos:.4f}  neg={_w_neg:.4f}  "
      f"ratio={_w_neg/_w_pos:.4f}")
if _w_neg / _w_pos < 0.02:
    print("  ⚠ WARNUNG: negatives Gewicht < 2% → Kollapsrisiko. gamma_neg senken!")


# ── Model ──────────────────────────────────────────────────────────────────
class TransformerClassifier(nn.Module):
    """
    gbert-large mit linearem Klassifikationskopf für Multilabel-Klassifikation.

    Hinweis: BertModel/BertTokenizer statt AutoModel/AutoTokenizer, da die
    config.json von deepset/gbert-large kein model_type-Feld enthält und die
    Auto-Klassen daher mit ValueError fehlschlagen.
    """

    def __init__(self, model_name, num_labels=4, dropout=0.2):
        super().__init__()
        self.encoder = BertModel.from_pretrained(
            model_name, ignore_mismatched_sizes=True
        )
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(
            self.encoder.config.hidden_size, num_labels
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kw["token_type_ids"] = token_type_ids

        out = self.encoder(**kw)
        pooled = (
            out.pooler_output
            if hasattr(out, "pooler_output") and out.pooler_output is not None
            else out.last_hidden_state[:, 0]
        )
        return self.classifier(self.dropout(pooled))


# ── Save config ────────────────────────────────────────────────────────────
cfg = {
    "experiment":    EXP_NAME,
    "note":          EXP_NOTE,
    "model_name":    MODEL_NAME,
    "mode":          MODE,
    "max_length":    MAX_LEN,
    "batch_size":    BATCH,
    "epochs":        EPOCHS,
    "patience":      PATIENCE,
    "learning_rate": LR,
    "warmup_ratio":  WARMUP_RATIO,
    "loss":          "MaskedASL",
    "asl_gamma_pos": ASL_GAMMA_POS,
    "asl_gamma_neg": ASL_GAMMA_NEG,
    "asl_clip":      ASL_CLIP,
    "label_weights": dict(zip(LABELS, LABEL_WEIGHTS)),
    "optimizer":     "AdamW",
    "weight_decay":  0.01,
    "seed":          42,
    "tokenizer":     "BertTokenizer",
    "encoder":       "BertModel",
    "reference":     "Ben-Baruch et al. (2021, ICCV) arXiv:2009.14119",
    "baseline_e7":   {"macro_s": E7_MACRO_S, "threat_s": E7_THREAT_S},
}
json.dump(cfg, open(f"{LOCAL_OUT}/config.json", "w"), indent=2, ensure_ascii=False)


# ── Drive sync ─────────────────────────────────────────────────────────────
def sync_to_drive():
    files = [
        "config.json", "summary_partial.json", "summary.json",
        "val_metrics_latest.csv", "val_tuned_latest.csv",
        "thresholds.json", "test_metrics.csv",
    ]
    for fname in files:
        src = f"{LOCAL_OUT}/{fname}"
        if os.path.exists(src):
            shutil.copy2(src, f"{DRIVE_OUT}/{fname}")

    for fname in os.listdir(LOCAL_OUT):
        if fname.startswith(("val_metrics_epoch_", "val_tuned_epoch_")) \
                and fname.endswith(".csv"):
            shutil.copy2(f"{LOCAL_OUT}/{fname}", f"{DRIVE_OUT}/{fname}")

    best_src = f"{LOCAL_OUT}/best_model"
    best_dst = f"{DRIVE_OUT}/best_model"
    if os.path.exists(best_src) and os.listdir(best_src):
        os.makedirs(best_dst, exist_ok=True)
        for fname in os.listdir(best_src):
            shutil.copy2(f"{best_src}/{fname}", f"{best_dst}/{fname}")

    if os.path.exists(CHECKPOINT_LOCAL):
        shutil.copy2(CHECKPOINT_LOCAL, CHECKPOINT_DRIVE)

    print("Synced to Drive.")


# ── Checkpoint ─────────────────────────────────────────────────────────────
def save_checkpoint(epoch, step):
    torch.save({
        "epoch":                epoch,
        "step":                 step,
        "model_state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_s":           best_val_s,
        "best_epoch":           best_epoch,
        "patience_left":        patience_left,
        "epoch_results":        epoch_results,
    }, CHECKPOINT_LOCAL)
    shutil.copy2(CHECKPOINT_LOCAL, CHECKPOINT_DRIVE)
    print(f"  Checkpoint saved: epoch={epoch}, step={step}")


# ── Data ───────────────────────────────────────────────────────────────────
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

splits = load_dataset(DATA_PATH, mode=MODE)
print_dataset_summary(splits)

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer: BertTokenizer, vocab_size={tokenizer.vocab_size}")


def make_loader(df, shuffle):
    ds = NetiquetteTransformerDataset(df, tokenizer, MAX_LEN)
    return DataLoader(
        ds, batch_size=BATCH, shuffle=shuffle,
        num_workers=0, pin_memory=True,
    )


train_loader = make_loader(splits["train"], shuffle=True)
val_loader   = make_loader(splits["val"],   shuffle=False)
test_loader  = make_loader(splits["test"],  shuffle=False)


# ── Model + Optimizer ──────────────────────────────────────────────────────
model     = TransformerClassifier(MODEL_NAME).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f"\n{'='*60}")
print(f"Experiment:  {EXP_NAME}")
print(f"Note:        {EXP_NOTE}")
print(f"Model:       {MODEL_NAME}, hidden_size={model.encoder.config.hidden_size}")
print(f"LR:          {LR}  (identisch zu E7)")
print(f"Steps/epoch: {len(train_loader):,} | Total: {total_steps:,} | Warmup: {warmup_steps:,}")
print(f"{'='*60}")


# ── Resume state ───────────────────────────────────────────────────────────
start_epoch   = 1
resume_step   = 0
best_val_s    = -1.0
best_epoch    = -1
patience_left = PATIENCE
epoch_results = []

if os.path.exists(CHECKPOINT_LOCAL):
    print(f"\nLoading checkpoint: {CHECKPOINT_LOCAL}")
    ckpt = torch.load(CHECKPOINT_LOCAL, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    start_epoch   = ckpt["epoch"]
    resume_step   = ckpt["step"]
    best_val_s    = ckpt.get("best_val_s",    -1.0)
    best_epoch    = ckpt.get("best_epoch",    -1)
    patience_left = ckpt.get("patience_left", PATIENCE)
    epoch_results = ckpt.get("epoch_results", [])
    print(f"Resuming from epoch={start_epoch}, step={resume_step}")
    print(f"Best so far: S={best_val_s:.4f} @ epoch {best_epoch} | patience={patience_left}")
else:
    print("\nStarting from scratch.")

sync_to_drive()


# ── Helper: collect logits ─────────────────────────────────────────────────
def collect_logits(loader):
    model.eval()
    all_logits, all_labels, all_masks = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(
                batch["input_ids"],
                batch["attention_mask"],
                batch.get("token_type_ids"),
            )
            all_logits.append(logits.cpu().numpy())
            all_labels.append(batch["labels"].cpu().numpy())
            all_masks.append(batch["label_mask"].cpu().numpy())
    return (
        np.concatenate(all_logits),
        np.concatenate(all_labels),
        np.concatenate(all_masks),
    )


# ── Collapse detection ─────────────────────────────────────────────────────
def check_collapse(metrics_df, epoch):
    """
    Erkennt Modellkollaps: MCC≈0 und Recall≈1 für alle Labels bedeutet,
    dass das Modell konstant die positive Klasse vorhersagt.
    """
    per_label = metrics_df[metrics_df["label"] != "MACRO"]
    all_mcc_zero    = (per_label["mcc"].abs() < 0.01).all()
    all_recall_one  = (per_label["recall"] > 0.99).all()

    if all_mcc_zero and all_recall_one:
        print(f"\n{'!'*60}")
        print(f"MODELLKOLLAPS ERKANNT (Epoch {epoch})")
        print(f"  MCC ≈ 0 und Recall ≈ 1 für alle Labels.")
        print(f"  Das Modell sagt konstant die positive Klasse vorher.")
        print(f"  → gamma_neg={ASL_GAMMA_NEG} zu hoch, oder LR={LR} zu groß.")
        print(f"  Training wird abgebrochen.")
        print(f"{'!'*60}\n")
        return True
    return False


# ── Training loop ──────────────────────────────────────────────────────────
collapsed = False

for epoch in range(start_epoch, EPOCHS + 1):

    if patience_left <= 0:
        print(f"\nEarly stopping before epoch {epoch}.")
        break

    print(f"\n{'='*60}")
    print(f"EPOCH {epoch}/{EPOCHS} | resume_step={resume_step} | patience={patience_left}")
    print(f"{'='*60}")

    model.train()
    total_loss = 0.0
    steps      = 0

    for step, batch in enumerate(train_loader, 1):

        if epoch == start_epoch and step <= resume_step:
            continue

        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        logits = model(
            batch["input_ids"],
            batch["attention_mask"],
            batch.get("token_type_ids"),
        )

        loss = loss_fn(logits, batch["labels"], batch["label_mask"])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        steps      += 1

        if step % 500 == 0:
            print(f"  step {step:>6}/{len(train_loader):,}  loss={total_loss/steps:.4f}")

        if step % SAVE_EVERY_STEPS == 0:
            save_checkpoint(epoch, step)

    if steps == 0:
        print(f"No new steps in epoch {epoch}. Skipping validation.")
        resume_step = 0
        continue

    train_loss = total_loss / steps
    print(f"\nEpoch {epoch} — train loss: {train_loss:.4f}")

    # ── Validation ─────────────────────────────────────────────────────────
    val_logits, val_labels, val_masks = collect_logits(val_loader)

    val_raw = compute_multilabel_metrics(
        val_logits, val_labels, val_masks, split_name="val"
    )
    print_metrics_table(f"VAL Epoch {epoch} (threshold=0.5)", val_raw)

    # Kollaps-Check nach Epoch 1 → früher Abbruch statt Zeitverschwendung
    if check_collapse(val_raw, epoch):
        collapsed = True
        json.dump({
            "experiment": EXP_NAME,
            "status":     "collapsed",
            "epoch":      epoch,
            "reason":     "MCC≈0 and Recall≈1 for all labels",
            "asl_gamma_neg": ASL_GAMMA_NEG,
            "learning_rate": LR,
        }, open(f"{LOCAL_OUT}/summary.json", "w"), indent=2, ensure_ascii=False)
        val_raw.to_csv(f"{LOCAL_OUT}/val_metrics_collapsed.csv", index=False)
        sync_to_drive()
        break

    tuned = tune_thresholds(val_logits, val_labels, val_masks, metric="s_score")
    print(f"\nTuned thresholds: {tuned}")

    val_tuned = compute_multilabel_metrics(
        val_logits, val_labels, val_masks,
        thresholds=tuned, split_name="val_tuned",
    )
    print_metrics_table(f"VAL Epoch {epoch} TUNED", val_tuned)

    macro_s = float(val_tuned[val_tuned["label"] == "MACRO"]["s_score"].iloc[0])

    val_raw.to_csv(f"{LOCAL_OUT}/val_metrics_epoch_{epoch}.csv", index=False)
    val_tuned.to_csv(f"{LOCAL_OUT}/val_tuned_epoch_{epoch}.csv", index=False)
    val_raw.to_csv(f"{LOCAL_OUT}/val_metrics_latest.csv",        index=False)
    val_tuned.to_csv(f"{LOCAL_OUT}/val_tuned_latest.csv",        index=False)

    epoch_results.append({
        "epoch":       epoch,
        "train_loss":  train_loss,
        "val_macro_s": macro_s,
    })

    # ── Early stopping ─────────────────────────────────────────────────────
    if macro_s > best_val_s:
        best_val_s    = macro_s
        best_epoch    = epoch
        patience_left = PATIENCE

        torch.save(model.state_dict(), f"{LOCAL_OUT}/best_model/pytorch_model.bin")
        tokenizer.save_pretrained(f"{LOCAL_OUT}/best_model")
        json.dump(tuned, open(f"{LOCAL_OUT}/thresholds.json", "w"), indent=2)

        print(f"\n★ New best: S={macro_s:.4f} (E7 baseline: {E7_MACRO_S:.4f}) "
              f"| patience reset to {PATIENCE}")
    else:
        patience_left -= 1
        print(f"\nNo improvement (best={best_val_s:.4f}). Patience left: {patience_left}")

    json.dump({
        "experiment":  EXP_NAME,
        "model_name":  MODEL_NAME,
        "mode":        MODE,
        "epoch":       epoch,
        "best_epoch":  best_epoch,
        "best_val_s":  best_val_s,
        "epochs_done": epoch_results,
        "status":      "training" if patience_left > 0 else "early_stopped",
    }, open(f"{LOCAL_OUT}/summary_partial.json", "w"), indent=2, ensure_ascii=False)

    resume_step = 0
    save_checkpoint(epoch + 1, 0)
    sync_to_drive()

    if patience_left <= 0:
        print(f"\nEarly stopping triggered at epoch {epoch}.")
        break


# ── Final Evaluation ───────────────────────────────────────────────────────
if collapsed:
    print("\nTraining abgebrochen wegen Modellkollaps. Keine finale Evaluation.")
    print(f"Empfehlung: gamma_neg auf 3 senken oder LR auf 2e-6 reduzieren.")
    raise SystemExit(0)

print(f"\n{'='*60}")
print(f"FINAL EVALUATION — Best epoch: {best_epoch} | val S={best_val_s:.4f}")
print(f"{'='*60}")

best_model_path = f"{LOCAL_OUT}/best_model/pytorch_model.bin"
if not os.path.exists(best_model_path):
    raise FileNotFoundError("No best model found. Training completed no epoch.")

model.load_state_dict(
    torch.load(best_model_path, map_location=device, weights_only=True)
)

# Threshold-Tuning auf Validierungsset
val_logits, val_labels, val_masks = collect_logits(val_loader)
best_thresholds = tune_thresholds(val_logits, val_labels, val_masks, metric="s_score")
print(f"Final thresholds: {best_thresholds}")
json.dump(best_thresholds, open(f"{LOCAL_OUT}/thresholds.json", "w"), indent=2)

np.savez(
    f"{LOCAL_OUT}/val_logits.npz",
    logits=val_logits, labels=val_labels, label_mask=val_masks,
)

# Testset
test_logits, test_labels, test_masks = collect_logits(test_loader)
test_metrics = compute_multilabel_metrics(
    test_logits, test_labels, test_masks,
    thresholds=best_thresholds, split_name="test",
)
print_metrics_table(f"FINAL TEST — {EXP_NAME}", test_metrics)

test_metrics.to_csv(f"{LOCAL_OUT}/test_metrics.csv", index=False)

np.savez(
    f"{LOCAL_OUT}/test_logits.npz",
    logits=test_logits, labels=test_labels, label_mask=test_masks,
)

# ── Vergleich E8 vs. E7 ────────────────────────────────────────────────────
macro_row  = test_metrics[test_metrics["label"] == "MACRO"].iloc[0]
threat_row = test_metrics[test_metrics["label"] == "threat"].iloc[0]

print(f"\n{'='*60}")
print(f"{EXP_NAME} (MaskedASL)  vs  E7 (Focal Loss)")
print(f"{'='*60}")
print(f"{'Metric':<18} {'E8':>10} {'E7':>10} {'Delta':>10}")
print(f"{'-'*50}")
print(f"{'Macro S-Score':<18} {macro_row['s_score']:>10.4f} {E7_MACRO_S:>10.4f} "
      f"{macro_row['s_score']-E7_MACRO_S:>+10.4f}")
print(f"{'Macro F1':<18} {macro_row['f1']:>10.4f} {E7_MACRO_F1:>10.4f} "
      f"{macro_row['f1']-E7_MACRO_F1:>+10.4f}")
print(f"{'Macro F2':<18} {macro_row['f2']:>10.4f} {E7_MACRO_F2:>10.4f} "
      f"{macro_row['f2']-E7_MACRO_F2:>+10.4f}")
print(f"{'threat S-Score':<18} {threat_row['s_score']:>10.4f} {E7_THREAT_S:>10.4f} "
      f"{threat_row['s_score']-E7_THREAT_S:>+10.4f}")
print(f"{'threat F1':<18} {threat_row['f1']:>10.4f} {E7_THREAT_F1:>10.4f} "
      f"{threat_row['f1']-E7_THREAT_F1:>+10.4f}")
print(f"{'='*60}")

json.dump({
    "experiment":    EXP_NAME,
    "note":          EXP_NOTE,
    "model_name":    MODEL_NAME,
    "mode":          MODE,
    "max_length":    MAX_LEN,
    "batch_size":    BATCH,
    "learning_rate": LR,
    "warmup_ratio":  WARMUP_RATIO,
    "loss":          "MaskedASL",
    "asl_gamma_pos": ASL_GAMMA_POS,
    "asl_gamma_neg": ASL_GAMMA_NEG,
    "asl_clip":      ASL_CLIP,
    "label_weights": dict(zip(LABELS, LABEL_WEIGHTS)),
    "best_epoch":    best_epoch,
    "best_val_s":    best_val_s,
    "epochs_done":   epoch_results,
    "status":        "done",
    "reference":     "Ben-Baruch et al. (2021, ICCV) arXiv:2009.14119",
    "test_macro":    macro_row.to_dict(),
    "test_threat":   threat_row.to_dict(),
    "comparison_e7": {
        "macro_s_delta":  round(float(macro_row["s_score"])  - E7_MACRO_S,  4),
        "macro_f1_delta": round(float(macro_row["f1"])       - E7_MACRO_F1, 4),
        "threat_s_delta": round(float(threat_row["s_score"]) - E7_THREAT_S, 4),
    },
}, open(f"{LOCAL_OUT}/summary.json", "w"), indent=2, ensure_ascii=False)

sync_to_drive()
print(f"\nDone! Results saved to:\n  {DRIVE_OUT}")

GPU:  Tesla T4
VRAM: 15.6 GB
Resume checkpoint found: /content/drive/MyDrive/masterarbeit/results/gbert_large_gold_silver_128_asl_v3/checkpoint_latest.pt

Loss: MaskedASL
  gamma_pos     = 0
  gamma_neg     = 4
  clip          = 0.05
  label_weights = {'hate_speech': 1.0, 'toxic': 1.0, 'threat': 1.0, 'insult': 1.0}
  Gradient weight @ p=0.5:  pos=1.0000  neg=0.0410  ratio=0.0410

Dataset summary:

TRAIN:
  rows   : 426,742
  gold   : 61,830
  silver : 364,912
  hate_speech  annotated=278,515 pos= 8,637 neg=269,878
  toxic        annotated=340,767 pos=34,936 neg=305,831
  threat       annotated=236,699 pos=   735 neg=235,964
  insult       annotated=286,314 pos=19,372 neg=266,942

VAL:
  rows   : 13,250
  gold   : 13,250
  silver : 0
  hate_speech  annotated= 13,250 pos= 1,422 neg= 11,828
  toxic        annotated=  2,722 pos=   819 neg=  1,903
  threat       annotated=  4,295 pos=    21 neg=  4,274
  insult       annotated=  7,017 pos= 1,296 neg=  5,721

TEST:
  rows   : 13,250
  gold  

tokenizer_config.json:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/240k [00:00<?, ?B/s]

Tokenizer: BertTokenizer, vocab_size=31102


config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: deepset/gbert-large
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Experiment:  E8
Note:        gbert-large gold_silver MaskedASL (ASL-Defaults: gamma_neg=4, clip=0.05)
Model:       deepset/gbert-large, hidden_size=1024
LR:          5e-06  (identisch zu E7)
Steps/epoch: 53,343 | Total: 266,715 | Warmup: 26,671

Loading checkpoint: results/gbert_large_gold_silver_128_asl_v3/checkpoint_latest.pt
Resuming from epoch=4, step=10000
Best so far: S=0.6464 @ epoch 3 | patience=2
Synced to Drive.

EPOCH 4/5 | resume_step=10000 | patience=2
  step  10500/53,343  loss=0.0234
  step  11000/53,343  loss=0.0233
  step  11500/53,343  loss=0.0226
  step  12000/53,343  loss=0.0232
  step  12500/53,343  loss=0.0230
  step  13000/53,343  loss=0.0231
  step  13500/53,343  loss=0.0230
  step  14000/53,343  loss=0.0229
  step  14500/53,343  loss=0.0231
  step  15000/53,343  loss=0.0232
  Checkpoint saved: epoch=4, step=15000
  step  15500/53,343  loss=0.0226
  step  16000/53,343  loss=0.0222
  step  16500/53,343  loss=0.0218
  step  17000/53,343  loss=0.0213
  step  17500

FileNotFoundError: No best model found. Training completed no epoch.

GPU:  Tesla T4
VRAM: 15.6 GB
Resume checkpoint found: /content/drive/MyDrive/masterarbeit/results/gbert_large_gold_silver_128_asl_v3/checkpoint_latest.pt

Loss: MaskedASL
  gamma_pos     = 0
  gamma_neg     = 4
  clip          = 0.05
  label_weights = {'hate_speech': 1.0, 'toxic': 1.0, 'threat': 1.0, 'insult': 1.0}
  Gradient weight @ p=0.5:  pos=1.0000  neg=0.0410  ratio=0.0410

Dataset summary:

TRAIN:
  rows   : 426,742
  gold   : 61,830
  silver : 364,912
  hate_speech  annotated=278,515 pos= 8,637 neg=269,878
  toxic        annotated=340,767 pos=34,936 neg=305,831
  threat       annotated=236,699 pos=   735 neg=235,964
  insult       annotated=286,314 pos=19,372 neg=266,942

VAL:
  rows   : 13,250
  gold   : 13,250
  silver : 0
  hate_speech  annotated= 13,250 pos= 1,422 neg= 11,828
  toxic        annotated=  2,722 pos=   819 neg=  1,903
  threat       annotated=  4,295 pos=    21 neg=  4,274
  insult       annotated=  7,017 pos= 1,296 neg=  5,721

TEST:
  rows   : 13,250
  gold   : 13,250
  silver : 0
  hate_speech  annotated= 13,250 pos= 1,422 neg= 11,828
  toxic        annotated=  2,722 pos=   819 neg=  1,903
  threat       annotated=  4,376 pos=    21 neg=  4,355
  insult       annotated=  7,098 pos= 1,296 neg=  5,802
tokenizer_config.json: 100%
 83.0/83.0 [00:00<00:00, 9.90kB/s]
Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
WARNING:huggingface_hub.utils._http:Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
vocab.txt: 100%
 240k/240k [00:00<00:00, 9.89MB/s]
Tokenizer: BertTokenizer, vocab_size=31102
config.json: 100%
 363/363 [00:00<00:00, 48.1kB/s]
model.safetensors: 100%
 1.35G/1.35G [00:13<00:00, 117MB/s]
Loading weights: 100%
 391/391 [00:00<00:00, 3209.30it/s]
[transformers] BertModel LOAD REPORT from: deepset/gbert-large
Key                                        | Status     |  |
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  |
cls.seq_relationship.weight                | UNEXPECTED |  |
cls.seq_relationship.bias                  | UNEXPECTED |  |
cls.predictions.transform.dense.bias       | UNEXPECTED |  |
cls.predictions.bias                       | UNEXPECTED |  |
cls.predictions.transform.dense.weight     | UNEXPECTED |  |
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  |

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.

============================================================
Experiment:  E8
Note:        gbert-large gold_silver MaskedASL (ASL-Defaults: gamma_neg=4, clip=0.05)
Model:       deepset/gbert-large, hidden_size=1024
LR:          5e-06  (identisch zu E7)
Steps/epoch: 53,343 | Total: 266,715 | Warmup: 26,671
============================================================

Loading checkpoint: results/gbert_large_gold_silver_128_asl_v3/checkpoint_latest.pt
Resuming from epoch=2, step=15000
Best so far: S=0.6144 @ epoch 1 | patience=2
Synced to Drive.

============================================================
EPOCH 2/5 | resume_step=15000 | patience=2
============================================================
  step  15500/53,343  loss=0.0284
  step  16000/53,343  loss=0.0291
  step  16500/53,343  loss=0.0292
  step  17000/53,343  loss=0.0288
  step  17500/53,343  loss=0.0287
  step  18000/53,343  loss=0.0283
  step  18500/53,343  loss=0.0287
  step  19000/53,343  loss=0.0285
  step  19500/53,343  loss=0.0285
  step  20000/53,343  loss=0.0284
  Checkpoint saved: epoch=2, step=20000
  step  20500/53,343  loss=0.0282
  step  21000/53,343  loss=0.0281
  step  21500/53,343  loss=0.0280
  step  22000/53,343  loss=0.0280
  step  22500/53,343  loss=0.0281
  step  23000/53,343  loss=0.0279
  step  23500/53,343  loss=0.0279
  step  24000/53,343  loss=0.0278
  step  24500/53,343  loss=0.0277
  step  25000/53,343  loss=0.0277
  Checkpoint saved: epoch=2, step=25000
  step  25500/53,343  loss=0.0275
  step  26000/53,343  loss=0.0275
  step  26500/53,343  loss=0.0274
  step  27000/53,343  loss=0.0275
  step  27500/53,343  loss=0.0275
  step  28000/53,343  loss=0.0275
  step  28500/53,343  loss=0.0275
  step  29000/53,343  loss=0.0276
  step  29500/53,343  loss=0.0275
  step  30000/53,343  loss=0.0275
  Checkpoint saved: epoch=2, step=30000
  step  30500/53,343  loss=0.0275
  step  31000/53,343  loss=0.0275
  step  31500/53,343  loss=0.0274
  step  32000/53,343  loss=0.0274
  step  32500/53,343  loss=0.0274
  step  33000/53,343  loss=0.0274
  step  33500/53,343  loss=0.0274
  step  34000/53,343  loss=0.0273
  step  34500/53,343  loss=0.0272
  step  35000/53,343  loss=0.0271
  Checkpoint saved: epoch=2, step=35000
  step  35500/53,343  loss=0.0271
  step  36000/53,343  loss=0.0270
  step  36500/53,343  loss=0.0269
  step  37000/53,343  loss=0.0269
  step  37500/53,343  loss=0.0270
  step  38000/53,343  loss=0.0270
  step  38500/53,343  loss=0.0270
  step  39000/53,343  loss=0.0269
  step  39500/53,343  loss=0.0269
  step  40000/53,343  loss=0.0269
  Checkpoint saved: epoch=2, step=40000
  step  40500/53,343  loss=0.0268
  step  41000/53,343  loss=0.0268
  step  41500/53,343  loss=0.0268
  step  42000/53,343  loss=0.0268
  step  42500/53,343  loss=0.0268
  step  43000/53,343  loss=0.0268
  step  43500/53,343  loss=0.0267
  step  44000/53,343  loss=0.0267
  step  44500/53,343  loss=0.0267
  step  45000/53,343  loss=0.0267
  Checkpoint saved: epoch=2, step=45000
  step  45500/53,343  loss=0.0267
  step  46000/53,343  loss=0.0266
  step  46500/53,343  loss=0.0266
  step  47000/53,343  loss=0.0265
  step  47500/53,343  loss=0.0265
  step  48000/53,343  loss=0.0265
  step  48500/53,343  loss=0.0265
  step  49000/53,343  loss=0.0265
  step  49500/53,343  loss=0.0265
  step  50000/53,343  loss=0.0265
  Checkpoint saved: epoch=2, step=50000
  step  50500/53,343  loss=0.0264
  step  51000/53,343  loss=0.0264
  step  51500/53,343  loss=0.0264
  step  52000/53,343  loss=0.0264
  step  52500/53,343  loss=0.0264
  step  53000/53,343  loss=0.0264

Epoch 2 — train loss: 0.0264

VAL Epoch 2 (threshold=0.5)
----------------------------------------------------------------------------------------------------
      label  threshold  precision  recall     f1     f2    mcc  s_score  support_pos  support_total
hate_speech     0.5000     0.3575  0.5992 0.4478 0.5278 0.3786   0.6085         1422          13250
      toxic     0.5000     0.6626  0.7912 0.7212 0.7616 0.5906   0.7785          819           2722
     threat     0.5000     0.2500  0.0952 0.1379 0.1087 0.1518   0.3423           21           4295
     insult     0.5000     0.5134  0.6343 0.5675 0.6057 0.4606   0.6680         1296           7017
      MACRO        NaN     0.4459  0.5300 0.4686 0.5010 0.3954   0.5993         3558          27284

Tuned thresholds: {'hate_speech': 0.45, 'toxic': 0.5, 'threat': 0.35000000000000003, 'insult': 0.45}

VAL Epoch 2 TUNED
----------------------------------------------------------------------------------------------------
      label  threshold  precision  recall     f1     f2    mcc  s_score  support_pos  support_total
hate_speech     0.4500     0.2910  0.7377 0.4173 0.5644 0.3628   0.6229         1422          13250
      toxic     0.5000     0.6626  0.7912 0.7212 0.7616 0.5906   0.7785          819           2722
     threat     0.3500     0.1818  0.3810 0.2462 0.3125 0.2581   0.4708           21           4295
     insult     0.4500     0.4467  0.7207 0.5515 0.6419 0.4399   0.6809         1296           7017
      MACRO        NaN     0.3955  0.6576 0.4841 0.5701 0.4128   0.6383         3558          27284

★ New best: S=0.6383 (E7 baseline: 0.6440) | patience reset to 2
  Checkpoint saved: epoch=3, step=0
Synced to Drive.

============================================================
EPOCH 3/5 | resume_step=0 | patience=2
============================================================
  step    500/53,343  loss=0.0203
  step   1000/53,343  loss=0.0204
  step   1500/53,343  loss=0.0208
  step   2000/53,343  loss=0.0208
  step   2500/53,343  loss=0.0203
  step   3000/53,343  loss=0.0202
  step   3500/53,343  loss=0.0199
  step   4000/53,343  loss=0.0199
  step   4500/53,343  loss=0.0197
  step   5000/53,343  loss=0.0197
  Checkpoint saved: epoch=3, step=5000
  step   5500/53,343  loss=0.0196
  step   6000/53,343  loss=0.0196
  step   6500/53,343  loss=0.0194
  step   7000/53,343  loss=0.0192
  step   7500/53,343  loss=0.0194
  step   8000/53,343  loss=0.0194
  step   8500/53,343  loss=0.0195
  step   9000/53,343  loss=0.0195
  step   9500/53,343  loss=0.0196
  step  10000/53,343  loss=0.0194
  Checkpoint saved: epoch=3, step=10000
  step  10500/53,343  loss=0.0196
  step  11000/53,343  loss=0.0196
  step  11500/53,343  loss=0.0197
  step  12000/53,343  loss=0.0197
  step  12500/53,343  loss=0.0198
  step  13000/53,343  loss=0.0198
  step  13500/53,343  loss=0.0198
  step  14000/53,343  loss=0.0198
  step  14500/53,343  loss=0.0198
  step  15000/53,343  loss=0.0199
  Checkpoint saved: epoch=3, step=15000
  step  15500/53,343  loss=0.0202
  step  16000/53,343  loss=0.0203
  step  16500/53,343  loss=0.0205
  step  17000/53,343  loss=0.0207
  step  17500/53,343  loss=0.0209
  step  18000/53,343  loss=0.0210
  step  18500/53,343  loss=0.0212
  step  19000/53,343  loss=0.0214
  step  19500/53,343  loss=0.0214
  step  20000/53,343  loss=0.0215
  Checkpoint saved: epoch=3, step=20000
  step  20500/53,343  loss=0.0216
  step  21000/53,343  loss=0.0217
  step  21500/53,343  loss=0.0218
  step  22000/53,343  loss=0.0219
  step  22500/53,343  loss=0.0220
  step  23000/53,343  loss=0.0222
  step  23500/53,343  loss=0.0222
  step  24000/53,343  loss=0.0223
  step  24500/53,343  loss=0.0223
  step  25000/53,343  loss=0.0224
  Checkpoint saved: epoch=3, step=25000
  step  25500/53,343  loss=0.0225
  step  26000/53,343  loss=0.0226
  step  26500/53,343  loss=0.0226
  step  27000/53,343  loss=0.0227
  step  27500/53,343  loss=0.0227
  step  28000/53,343  loss=0.0228
  step  28500/53,343  loss=0.0228
  step  29000/53,343  loss=0.0229
  step  29500/53,343  loss=0.0229
  step  30000/53,343  loss=0.0230
  Checkpoint saved: epoch=3, step=30000
  step  30500/53,343  loss=0.0230
  step  31000/53,343  loss=0.0230
  step  31500/53,343  loss=0.0231
  step  32000/53,343  loss=0.0231
  step  32500/53,343  loss=0.0232
  step  33000/53,343  loss=0.0232
  step  33500/53,343  loss=0.0232
  step  34000/53,343  loss=0.0233
  step  34500/53,343  loss=0.0233
  step  35000/53,343  loss=0.0234
  Checkpoint saved: epoch=3, step=35000
  step  35500/53,343  loss=0.0234
  step  36000/53,343  loss=0.0234
  step  36500/53,343  loss=0.0235
  step  37000/53,343  loss=0.0236
  step  37500/53,343  loss=0.0236
  step  38000/53,343  loss=0.0237
  step  38500/53,343  loss=0.0237
  step  39000/53,343  loss=0.0237
  step  39500/53,343  loss=0.0238
  step  40000/53,343  loss=0.0238
  Checkpoint saved: epoch=3, step=40000
  step  40500/53,343  loss=0.0239
  step  41000/53,343  loss=0.0239
  step  41500/53,343  loss=0.0240
  step  42000/53,343  loss=0.0240
  step  42500/53,343  loss=0.0240
  step  43000/53,343  loss=0.0240
  step  43500/53,343  loss=0.0240
  step  44000/53,343  loss=0.0240
  step  44500/53,343  loss=0.0240
  step  45000/53,343  loss=0.0240
  Checkpoint saved: epoch=3, step=45000
  step  45500/53,343  loss=0.0241
  step  46000/53,343  loss=0.0241
  step  46500/53,343  loss=0.0241
  step  47000/53,343  loss=0.0242
  step  47500/53,343  loss=0.0242
  step  48000/53,343  loss=0.0242
  step  48500/53,343  loss=0.0242
  step  49000/53,343  loss=0.0242
  step  49500/53,343  loss=0.0243
  step  50000/53,343  loss=0.0243
  Checkpoint saved: epoch=3, step=50000
  step  50500/53,343  loss=0.0243
  step  51000/53,343  loss=0.0243
  step  51500/53,343  loss=0.0243
  step  52000/53,343  loss=0.0243
  step  52500/53,343  loss=0.0244
  step  53000/53,343  loss=0.0244

Epoch 3 — train loss: 0.0244

VAL Epoch 3 (threshold=0.5)
----------------------------------------------------------------------------------------------------
      label  threshold  precision  recall     f1     f2    mcc  s_score  support_pos  support_total
hate_speech     0.5000     0.3920  0.5478 0.4570 0.5075 0.3864   0.6003         1422          13250
      toxic     0.5000     0.6580  0.8034 0.7235 0.7694 0.5934   0.7830          819           2722
     threat     0.5000     0.1818  0.0952 0.1250 0.1053 0.1285   0.3348           21           4295
     insult     0.5000     0.4778  0.7045 0.5694 0.6434 0.4620   0.6872         1296           7017
      MACRO        NaN     0.4274  0.5377 0.4687 0.5064 0.3926   0.6013         3558          27284

Tuned thresholds: {'hate_speech': 0.4, 'toxic': 0.45, 'threat': 0.4, 'insult': 0.45}

VAL Epoch 3 TUNED
----------------------------------------------------------------------------------------------------
      label  threshold  precision  recall     f1     f2    mcc  s_score  support_pos  support_total
hate_speech     0.4000     0.2744  0.7750 0.4053 0.5678 0.3560   0.6229         1422          13250
      toxic     0.4500     0.6280  0.8388 0.7182 0.7860 0.5846   0.7892          819           2722
     threat     0.4000     0.2162  0.3810 0.2759 0.3306 0.2824   0.4859           21           4295
     insult     0.4500     0.4167  0.7724 0.5414 0.6598 0.4314   0.6877         1296           7017
      MACRO        NaN     0.3838  0.6918 0.4852 0.5860 0.4136   0.6464         3558          27284

★ New best: S=0.6464 (E7 baseline: 0.6440) | patience reset to 2
  Checkpoint saved: epoch=4, step=0
Synced to Drive.

============================================================
EPOCH 4/5 | resume_step=0 | patience=2
============================================================
  step    500/53,343  loss=0.0176
  step   1000/53,343  loss=0.0171
  step   1500/53,343  loss=0.0176
  step   2000/53,343  loss=0.0179
  step   2500/53,343  loss=0.0181
  step   3000/53,343  loss=0.0179
  step   3500/53,343  loss=0.0182
  step   4000/53,343  loss=0.0181
  step   4500/53,343  loss=0.0182
  step   5000/53,343  loss=0.0182
  Checkpoint saved: epoch=4, step=5000
  step   5500/53,343  loss=0.0184
  step   6000/53,343  loss=0.0185
  step   6500/53,343  loss=0.0183
  step   7000/53,343  loss=0.0185
  step   7500/53,343  loss=0.0185
  step   8000/53,343  loss=0.0185
  step   8500/53,343  loss=0.0184
  step   9000/53,343  loss=0.0185
  step   9500/53,343  loss=0.0186
  step  10000/53,343  loss=0.0186
  Checkpoint saved: epoch=4, step=10000
  step  10500/53,343  loss=0.0185
  step  11000/53,343  loss=0.0184

In [ ]:
# 1. Lưu run collapse lại làm ablation evidence cho Kapitel 6
!cp -r /content/drive/MyDrive/masterarbeit/results/gbert_large_gold_silver_128_asl_v2 \
       /content/drive/MyDrive/masterarbeit/results/E8_ablation_collapsed_gamma6

# 2. Xóa run cũ
!rm -rf /content/drive/MyDrive/masterarbeit/results/gbert_large_gold_silver_128_asl_v2
!rm -rf results/gbert_large_gold_silver_128_asl_v2

In [ ]:
import os, shutil, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import BertModel, BertTokenizer

from src.training.transformer_dataset import NetiquetteTransformerDataset, load_dataset
from src.training.transformer_metrics import compute_multilabel_metrics, tune_thresholds, print_metrics_table

RUN_ID    = "gbert_large_gold_silver_128_asl_v3"
LOCAL_OUT = f"results/{RUN_ID}"
DRIVE_OUT = f"/content/drive/MyDrive/masterarbeit/results/{RUN_ID}"
DATA_PATH = "/content/drive/MyDrive/masterarbeit/unified_final_v1.parquet"
MODEL_NAME = "deepset/gbert-large"
MAX_LEN, BATCH = 128, 8
device = torch.device("cuda")

# Copy best_model từ Drive về local
os.makedirs(f"{LOCAL_OUT}/best_model", exist_ok=True)
for fname in os.listdir(f"{DRIVE_OUT}/best_model"):
    shutil.copy2(f"{DRIVE_OUT}/best_model/{fname}", f"{LOCAL_OUT}/best_model/{fname}")
print("Best model (Epoch 3) restored from Drive.")

class TransformerClassifier(nn.Module):
    def __init__(self, model_name, num_labels=4, dropout=0.2):
        super().__init__()
        self.encoder = BertModel.from_pretrained(model_name, ignore_mismatched_sizes=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_labels)
    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kw["token_type_ids"] = token_type_ids
        out = self.encoder(**kw)
        pooled = (out.pooler_output if hasattr(out, "pooler_output") and out.pooler_output is not None
                  else out.last_hidden_state[:, 0])
        return self.classifier(self.dropout(pooled))

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = TransformerClassifier(MODEL_NAME).to(device)
model.load_state_dict(torch.load(f"{LOCAL_OUT}/best_model/pytorch_model.bin",
                                 map_location=device, weights_only=True))
model.eval()
print("Model loaded.")

splits = load_dataset(DATA_PATH, mode="gold_silver")
def make_loader(df):
    return DataLoader(NetiquetteTransformerDataset(df, tokenizer, MAX_LEN),
                      batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)
val_loader, test_loader = make_loader(splits["val"]), make_loader(splits["test"])

def collect(loader):
    ll, lb, lm = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(batch["input_ids"], batch["attention_mask"], batch.get("token_type_ids"))
            ll.append(logits.cpu().numpy()); lb.append(batch["labels"].cpu().numpy()); lm.append(batch["label_mask"].cpu().numpy())
    return np.concatenate(ll), np.concatenate(lb), np.concatenate(lm)

# Tune thresholds trên val
print("Val inference...")
val_logits, val_labels, val_masks = collect(val_loader)
best_thresholds = tune_thresholds(val_logits, val_labels, val_masks, metric="s_score")
print("Thresholds:", best_thresholds)
json.dump(best_thresholds, open(f"{LOCAL_OUT}/thresholds.json", "w"), indent=2)
np.savez(f"{LOCAL_OUT}/val_logits.npz", logits=val_logits, labels=val_labels, label_mask=val_masks)

# Test
print("Test inference...")
test_logits, test_labels, test_masks = collect(test_loader)
test_metrics = compute_multilabel_metrics(test_logits, test_labels, test_masks,
                                          thresholds=best_thresholds, split_name="test")
print_metrics_table("FINAL TEST — E8", test_metrics)
test_metrics.to_csv(f"{LOCAL_OUT}/test_metrics.csv", index=False)
np.savez(f"{LOCAL_OUT}/test_logits.npz", logits=test_logits, labels=test_labels, label_mask=test_masks)

# So sánh E8 vs E7
macro  = test_metrics[test_metrics["label"] == "MACRO"].iloc[0]
threat = test_metrics[test_metrics["label"] == "threat"].iloc[0]
E7 = {"macro_s": 0.6440, "macro_f1": 0.4930, "threat_s": 0.4770, "threat_f1": 0.2860}
print(f"\n{'='*56}\nE8 (MaskedASL) vs E7 (Focal Loss)\n{'='*56}")
print(f"{'Metric':<16}{'E8':>10}{'E7':>10}{'Delta':>10}")
print(f"{'-'*46}")
print(f"{'Macro S':<16}{macro['s_score']:>10.4f}{E7['macro_s']:>10.4f}{macro['s_score']-E7['macro_s']:>+10.4f}")
print(f"{'Macro F1':<16}{macro['f1']:>10.4f}{E7['macro_f1']:>10.4f}{macro['f1']-E7['macro_f1']:>+10.4f}")
print(f"{'threat S':<16}{threat['s_score']:>10.4f}{E7['threat_s']:>10.4f}{threat['s_score']-E7['threat_s']:>+10.4f}")
print(f"{'threat F1':<16}{threat['f1']:>10.4f}{E7['threat_f1']:>10.4f}{threat['f1']-E7['threat_f1']:>+10.4f}")
print(f"{'='*56}")

# Sync
for f in ["thresholds.json", "test_metrics.csv", "test_logits.npz", "val_logits.npz"]:
    shutil.copy2(f"{LOCAL_OUT}/{f}", f"{DRIVE_OUT}/{f}")
print("Synced to Drive.")

Best model (Epoch 3) restored from Drive.


tokenizer_config.json:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/240k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: deepset/gbert-large
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.
Val inference...
Thresholds: {'hate_speech': 0.4, 'toxic': 0.45, 'threat': 0.4, 'insult': 0.45}
Test inference...

FINAL TEST — E8
----------------------------------------------------------------------------------------------------
      label  threshold  precision  recall     f1     f2    mcc  s_score  support_pos  support_total
hate_speech     0.4000     0.2750  0.7707 0.4054 0.5665 0.3554   0.6221         1422          13250
      toxic     0.4500     0.6271  0.8315 0.7150 0.7806 0.5795   0.7852          819           2722
     threat     0.4000     0.1000  0.1429 0.1176 0.1316 0.1145   0.3444           21           4376
     insult     0.4500     0.4294  0.7809 0.5541 0.6710 0.4504   0.6981         1296           7098
      MACRO        NaN     0.3579  0.6315 0.4480 0.5374 0.3749   0.6124         3558          27446

E8 (MaskedASL) vs E7 (Focal Loss)
Metric                  E8        E7     Delta
----------------------------------------------
Macro S             0.612